# Three-Month Research Progress Report

## Multimodal Disaster Understanding using Reliability and Uncertainty-Aware Learning

**Dataset:** CrisisMMD v2.0 | **Report notebook derived from:** `baseline (1)(1).ipynb`

---

**Note on scope.** This notebook is a reorganized, report-oriented derivative of the
existing baseline-comparison notebook. It reuses the existing dataset pipeline,
encoders, and architecture modules **verbatim**, and reports **only** the results
that were actually produced by that notebook's execution. No result has been
recalculated, re-estimated, or improved. The original notebook (`baseline (1)(1).ipynb`)
is left unmodified.

**B5 (the full integrated proposed model) is excluded from this report.** In the
source notebook, B5 was not retrained but only loaded from an existing checkpoint
if available; at the time this notebook was produced, that checkpoint/metrics file
was not found, so **B5 status = `NR` (not reported)** in the source notebook itself.
B5 is therefore reserved for the subsequent research stage and is not part of the
current three-month evaluation, consistent with the reporting scope for this stage.

**Current progress-report scope:** `B0` (Conventional Fusion) · `B1` (Vanilla EDL) ·
`B2` (EDL + UASG) · `B3` (E-REM + UASG) · `B4` (E-REM + Conventional Attention).

## Table of Contents

1. Research Objective
2. Dataset and Data Preparation
3. Baseline Development
4. Multimodal Representation and Uncertainty Investigation
5. Progressive Component Development
6. B0-B4 Experimental Evaluation
7. Training and Validation Analysis
8. Error Analysis
9. Uncertainty and Calibration Analysis
10. Qualitative Analysis
11. Summary of Three-Month Progress
12. Current Limitations and Next Research Stage

## 1. Research Objective

This research develops a **reliability-aware multimodal disaster-understanding
system** on CrisisMMD v2.0, combining a DeBERTa-v3 text encoder and a CLIP visual
encoder across five tasks: Event Type, Informativeness, Humanitarian category,
Damage Severity, and (cross-modal) Verification.

The central research question addressed in this three-month evaluation:

> Does the proposed **E-REM** (evidential reliability estimation module) provide
> additional benefit over a conventional evidential-uncertainty baseline
> (**Vanilla EDL**), and does propagating the resulting reliability signal through
> **UASG** (uncertainty-aware semantic gating) and cross-modal attention improve
> multimodal disaster understanding relative to a conventional fusion baseline?

To isolate the contribution of each proposed component, five configurations are
compared under an identical dataset split, preprocessing pipeline, and evaluation
protocol:

| Config | Description | E-REM | UASG | Cross-modal attention |
|---|---|---|---|---|
| B0 | Conventional Fusion | No | No | No |
| B1 | Vanilla EDL | No (plain EDL) | No | No |
| B2 | EDL + UASG | No (plain EDL) | Yes | No |
| B3 | E-REM + UASG | Yes | Yes | No |
| B4 | E-REM + Conventional Attention | Yes | Yes | Conventional |

The full integrated configuration (**B5**, adding InfoNCE alignment and UG-CMA) is
reserved for the next research stage and is **not** included in this report.

## 2. Dataset and Data Preparation

### Month 1 evidence — dataset pipeline, split, and preprocessing (reused verbatim below).

## SECTION 1 — Environment Setup (identical to Notebook 1)

In [ ]:
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

pip_install('sentencepiece')
pip_install('protobuf')
pip_install('transformers==4.40.1')
pip_install('open_clip_torch')

print('\u2705 All packages installed successfully.')

import gc
gc.collect()


In [ ]:
import os, sys, warnings, logging, random, json, time, copy, math, gc, pickle, shutil
from pathlib import Path
from datetime import datetime
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.ERROR)

import numpy  as np
import pandas as pd
from scipy.stats import entropy as scipy_entropy

import matplotlib
import matplotlib.pyplot  as plt
import seaborn as sns
matplotlib.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
})

import torch
import torch.nn            as nn
import torch.nn.functional as F
import torch.optim         as optim
from torch.utils.data      import Dataset, DataLoader
from torch.cuda.amp        import GradScaler, autocast

from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
import open_clip

from PIL         import Image
import cv2
from torchvision  import transforms

from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score, accuracy_score,
    precision_recall_fscore_support, precision_score, recall_score,
    cohen_kappa_score, matthews_corrcoef, roc_auc_score, average_precision_score,
    brier_score_loss,
)
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.model_selection import train_test_split

def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 42
seed_everything(SEED)
print(f'\u2705 All libraries imported | Seed fixed to {SEED}')
free_memory()


In [ ]:
if torch.cuda.is_available():
    DEVICE     = torch.device('cuda')
    N_GPUS     = torch.cuda.device_count()
    GPU_NAMES  = [torch.cuda.get_device_name(i) for i in range(N_GPUS)]
    VRAM_GB    = [torch.cuda.get_device_properties(i).total_memory / 1e9 for i in range(N_GPUS)]
else:
    DEVICE    = torch.device('cpu')
    N_GPUS    = 0
    GPU_NAMES = []
    VRAM_GB   = []

USE_AMP = N_GPUS > 0

print('=' * 60)
print('  \U0001F5A5\uFE0F  Hardware Configuration')
print('=' * 60)
print(f'  PyTorch Version  : {torch.__version__}')
print(f'  CUDA Available   : {torch.cuda.is_available()}')
print(f'  Device           : {DEVICE}')
print(f'  Number of GPUs   : {N_GPUS}')
for i, (name, vram) in enumerate(zip(GPU_NAMES, VRAM_GB)):
    print(f'    GPU {i}: {name}  ({vram:.1f} GB VRAM)')
print(f'  Mixed Precision  : {USE_AMP}')
print('=' * 60)
free_memory()


## SECTION 2 — Global Configuration (identical `CFG` to Notebook 1)

In [ ]:
def _find_dataset_root():
    env = os.environ.get('CRISISMD_ROOT')
    if env and Path(env).exists():
        return Path(env)
    kaggle_slugs = [
        '/kaggle/input/datasets/seaninggg/crisismmd-multimodal-crisis-dataset',
        '/kaggle/input/datasets/seaninggg/crisismmd-multimodal-crisis-dataset/CrisisMMD_v2.0',
        '/kaggle/input/crisismmd-multimodal-crisis-dataset',
        '/kaggle/input/crisismmdv2',
        '/kaggle/input/crisis-mmd',
    ]
    for s in kaggle_slugs:
        if Path(s).exists():
            return Path(s)
    local_tries = [
        Path('/content/CrisisMMD_v2.0'), Path('/content/CrisisMMD'),
        Path('/workspace/CrisisMMD'), Path('/data/CrisisMMD'),
        Path.home() / 'datasets' / 'CrisisMMD', Path.home() / 'data' / 'CrisisMMD',
        Path('./CrisisMMD_v2.0'), Path('./CrisisMMD'),
    ]
    for p in local_tries:
        if p.exists():
            return p
    raise FileNotFoundError(
        '\n' + '=' * 65 + '\nCrisisMMD dataset NOT FOUND.\n\n'
        'Check the right-hand sidebar under "Data" and ensure the dataset is attached.\n'
        + '=' * 65
    )

def _find_data_dir(root):
    sub = root / 'CrisisMMD_v2.0'
    return sub if sub.exists() else root

def _find_output_dir():
    for candidate in [Path('/kaggle/working'), Path('/content'), Path('./outputs')]:
        if candidate.parent.exists():
            candidate.mkdir(parents=True, exist_ok=True)
            return candidate
    return Path('.')

try:
    _ROOT = _find_dataset_root()
except FileNotFoundError as e:
    print(e)
    _ROOT = Path('.')

_DATA    = _find_data_dir(_ROOT)
_ANN_DIR = _DATA / 'annotations'
_IMG_DIR = _DATA / 'data_image'
_OUT_DIR = _find_output_dir()

class CFG:
    ROOT      = _ROOT
    DATA_DIR  = _DATA
    ANN_DIR   = _ANN_DIR
    IMG_DIR   = _IMG_DIR
    OUT_DIR   = _OUT_DIR

    ALL_TSVS  = [
        f for f in _ANN_DIR.iterdir()
        if f.suffix == '.tsv' and not f.name.startswith('._')
    ] if _ANN_DIR.exists() else []

    ROBERTA_CKPT  = 'microsoft/deberta-v3-base'
    CLIP_MODEL    = 'ViT-B-32'
    CLIP_PRETRAIN = 'openai'

    TEXT_DIM      = 768
    VIS_DIM       = 768
    FUSE_DIM      = 768
    HEAD_DIM      = 256

    BATCH_SIZE    = 32
    LR_ROBERTA    = 1e-5
    LR_CLIP       = 1e-5
    LR_HEAD       = 3e-4
    WEIGHT_DECAY  = 1e-2
    WARMUP_RATIO  = 0.1
    MAX_GRAD_NORM = 1.0
    MAX_LEN       = 128
    IMG_SIZE      = 224
    PATIENCE      = 5

    CKPT_TASKS       = ['human', 'damage']
    FOCAL_GAMMA_HUMAN  = 1.5
    FOCAL_GAMMA_DAMAGE = 1.2
    USE_SAMPLER      = True
    USE_CLASS_BALANCED_LOSS = True
    CB_BETA                 = 0.999
    USE_LOGIT_CALIBRATION   = False   # kept simple / off for baselines; B5 uses its own existing calibration
    USE_TTA                 = False
    USE_NO_DECAY_GROUPS   = True
    LOG_VAR_CLAMP         = (-3.0, 3.0)
    GRAD_ACCUM_STEPS      = 2
    FREEZE_ENCODER_EPOCHS = 2

    HUMAN_DAMAGE_LABEL_SMOOTHING = 0.08

    LAMBDA_EVENT   = 1.0
    LAMBDA_INFO    = 1.2
    LAMBDA_HUMAN   = 2.0
    LAMBDA_DAMAGE  = 2.0
    LAMBDA_VERIF   = 1.2

    FOCAL_GAMMA_VERIF = 1.5
    CB_BETA_VERIF     = 0.99
    VERIF_LABEL_SMOOTHING = 0.05

    EREM_N_PROTOTYPES = 8

    EVENT_LABELS  = ['earthquake', 'fire', 'flood', 'hurricane']
    INFO_LABELS   = ['informative', 'not_informative']
    HUMAN_LABELS  = [
        'affected_individuals', 'infrastructure_and_utility_damage',
        'not_humanitarian', 'other_relevant_information',
        'rescue_volunteering_or_donation_effort',
        'vehicle_damage', 'missing_or_found_people'
    ]
    DAMAGE_LABELS = ['little_or_no_damage', 'mild_damage', 'severe_damage']
    VERIF_LABELS  = ['authentic', 'fabricated']

    N_EVENT   = len(EVENT_LABELS)
    N_INFO    = len(INFO_LABELS)
    N_HUMAN   = len(HUMAN_LABELS)
    N_DAMAGE  = len(DAMAGE_LABELS)
    N_VERIF   = len(VERIF_LABELS)

    CKPT_DIR    = _OUT_DIR / 'checkpoints'
    PLOTS_DIR   = _OUT_DIR / 'plots'
    METRICS_DIR = _OUT_DIR / 'metrics'
    PRED_DIR    = _OUT_DIR / 'predictions'
    for _d in (CKPT_DIR, PLOTS_DIR, METRICS_DIR, PRED_DIR):
        _d.mkdir(parents=True, exist_ok=True)

    CKPT_PATH = str(CKPT_DIR / 'best_model.pt')   # existing Notebook-1 full-model checkpoint

cfg = CFG()

# --- Baseline-comparison specific paths (kept SEPARATE from cfg.CKPT_PATH / best_model.pt) ---
RESULTS_DIR      = 'results'
CKPT_SAVE_DIR    = os.path.join(RESULTS_DIR, 'checkpoints')     # existing Notebook-1 artifacts live here
DATA_SAVE_DIR    = os.path.join(RESULTS_DIR, 'data')
METRICS_SAVE_DIR = os.path.join(RESULTS_DIR, 'metrics')

BASELINE_CKPT_DIR = os.path.join(RESULTS_DIR, 'checkpoints')    # b0..b4 checkpoints saved alongside, NEVER overwriting best_model.pt
BASELINE_METRICS_DIR = os.path.join(RESULTS_DIR, 'baseline_metrics')
BASELINE_PLOTS_DIR   = str(cfg.PLOTS_DIR)
for _d in (BASELINE_CKPT_DIR, BASELINE_METRICS_DIR):
    os.makedirs(_d, exist_ok=True)

print('\U0001F4CB Configuration Summary')
print('\u2500' * 70)
print(f'  Root Base   : {cfg.ROOT}')
print(f'  Annotations : {cfg.ANN_DIR}')
print(f'  TSVs Found  : {len(cfg.ALL_TSVS)}')
print(f'  Notebook-1 checkpoints dir : {CKPT_SAVE_DIR}')
print(f'  Notebook-1 data dir        : {DATA_SAVE_DIR}')
print(f'  Baseline checkpoints -> b0_conventional_fusion.pt, b1_vanilla_edl.pt, b2_edl_uasg.pt,')
print(f'                          b3_erem_uasg.pt, b4_erem_conventional_attention.pt')
print(f'  NOTE: results/checkpoints/best_model.pt (existing B5 checkpoint) is NEVER overwritten.')
print('\u2500' * 70)
free_memory()


## SECTION 3 — Dataset Loading: reuse the EXACT existing `df_train` / `df_dev` / `df_test`

This notebook never creates a new split. Two paths are supported, in this priority order:

1. **Preferred:** if `results/data/{df_train,df_dev,df_test}.pkl` were saved by Notebook 1's
   Section 20.5 cell, load them directly — these are the *exact same objects* used to train
   the existing full model.
2. **Fallback** (only if artifacts weren't saved, e.g. running in the same live kernel session
   right after Notebook 1, or artifacts are missing): rebuild the split using the *identical*
   loading code, TSV read order, and `random_state=42` `train_test_split` calls as Notebook 1's
   Section 2 — this is deterministic and reproduces the same split bit-for-bit, but is not the
   literal same object, so it is used only as a fallback and is clearly logged as such.

Either way, dataset sizes are printed before every experiment (Section 4 & Section 23 below).


In [ ]:
_loaded_from_pickle = False
try:
    df_train = pd.read_pickle(os.path.join(DATA_SAVE_DIR, 'df_train.pkl'))
    df_dev   = pd.read_pickle(os.path.join(DATA_SAVE_DIR, 'df_dev.pkl'))
    df_test  = pd.read_pickle(os.path.join(DATA_SAVE_DIR, 'df_test.pkl'))
    with open(os.path.join(DATA_SAVE_DIR, 'label_encoders.pkl'), 'rb') as f:
        LABEL_ENCODERS = pickle.load(f)
    _loaded_from_pickle = True
    print('\u2705 Loaded df_train / df_dev / df_test / LABEL_ENCODERS from results/data/*.pkl '
          '(exact artifacts saved by Notebook 1, Section 20.5).')
except FileNotFoundError:
    print('\u26A0\uFE0F  results/data/*.pkl not found (Notebook 1 artifacts not saved, or this is '
          'a fresh session). Falling back to an IDENTICAL, deterministic rebuild of the split '
          '(same TSV read order + random_state=42) so df_train/df_dev/df_test are reproduced '
          'bit-for-bit -- but note this is a rebuild, not the literal saved object.')

    print('Loading and combining all event TSVs...')
    dfs = []
    for tsv_path in cfg.ALL_TSVS:
        try:
            df = pd.read_csv(tsv_path, sep='\t', on_bad_lines='skip')
            df.columns = [c.strip().lower() for c in df.columns]
            if 'event_name' not in df.columns and 'event_type' not in df.columns:
                name = tsv_path.stem.lower()
                if 'earthquake' in name: df['event_type'] = 'earthquake'
                elif 'fire' in name: df['event_type'] = 'fire'
                elif 'flood' in name: df['event_type'] = 'flood'
                else: df['event_type'] = 'hurricane'
            dfs.append(df)
        except Exception:
            pass

    full_df = pd.concat(dfs, ignore_index=True)
    df_train, temp_df = train_test_split(full_df, test_size=0.2, random_state=42, stratify=full_df['event_type'])
    df_dev, df_test   = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['event_type'])

    RENAME_MAP = {
        'tweet_text': 'text', 'image': 'image_path', 'label_image': 'label_image',
        'label_text': 'label_text', 'label': 'label_text', 'event_type': 'event_type',
        'event_name': 'event_type', 'humanitarian_class': 'humanitarian',
        'text_human': 'humanitarian', 'damage_type': 'damage', 'image_damage': 'damage',
        'text_info': 'informative', 'informativeness': 'informative',
    }

    def normalize_cols(df):
        df = df.copy()
        df = df.rename(columns={k: v for k, v in RENAME_MAP.items() if k in df.columns})
        if 'informative' not in df.columns and 'label_text' in df.columns:
            df['informative'] = df['label_text']
        if 'image_path' in df.columns:
            def resolve_path(p):
                p_str = str(p).strip()
                if p_str.startswith('/'):
                    return p_str
                if p_str.startswith('data_image/'):
                    return str(cfg.DATA_DIR / p_str)
                return str(cfg.IMG_DIR / p_str)
            df['image_path'] = df['image_path'].apply(resolve_path)
        if 'humanitarian' in df.columns:
            df['humanitarian'] = df['humanitarian'].replace({
                'missing_or_found_people': 'other_relevant_information',
                'vehicle_damage': 'other_relevant_information'
            })
        return df

    df_train = normalize_cols(df_train)
    df_dev   = normalize_cols(df_dev)
    df_test  = normalize_cols(df_test)

    def encode_label_col(df_t, df_d, df_te, col):
        le = LabelEncoder()
        all_vals = pd.concat([df_t[col], df_d[col], df_te[col]]).dropna().unique()
        le.fit(all_vals)
        for df in [df_t, df_d, df_te]:
            df[f'{col}_idx'] = le.transform(df[col].fillna(le.classes_[0]))
        return le

    LABEL_ENCODERS = {}
    for col in ['event_type', 'informative', 'humanitarian', 'damage']:
        if col in df_train.columns:
            LABEL_ENCODERS[col] = encode_label_col(df_train, df_dev, df_test, col)

    _IMAGE_INFO_CANDIDATES = ['image_info', 'img_info', 'image_informative', 'label_image_info']
    _img_info_col = next((c for c in _IMAGE_INFO_CANDIDATES if c in df_train.columns), None)
    if _img_info_col is not None:
        def _build_verif_label(df):
            df = df.copy()
            txt = df['informative'].astype(str).str.strip().str.lower()
            img = df[_img_info_col].astype(str).str.strip().str.lower()
            both_present = txt.notna() & img.notna() & (txt != 'nan') & (img != 'nan')
            agree = (txt == img)
            df['verif_idx'] = np.where(both_present, agree.astype(int), np.nan)
            return df
        df_train = _build_verif_label(df_train)
        df_dev   = _build_verif_label(df_dev)
        df_test  = _build_verif_label(df_test)
        VERIF_IS_REAL = True
    else:
        for _df in (df_train, df_dev, df_test):
            _df['verif_idx'] = (_df['informative'].astype(str).str.strip().str.lower() == 'not_informative').astype(int)
        VERIF_IS_REAL = False

    globals()['VERIF_IS_REAL'] = VERIF_IS_REAL

if 'event_type' in LABEL_ENCODERS: cfg.N_EVENT = len(LABEL_ENCODERS['event_type'].classes_)
if 'informative' in LABEL_ENCODERS: cfg.N_INFO = len(LABEL_ENCODERS['informative'].classes_)
if 'humanitarian' in LABEL_ENCODERS: cfg.N_HUMAN = len(LABEL_ENCODERS['humanitarian'].classes_)
if 'damage' in LABEL_ENCODERS: cfg.N_DAMAGE = len(LABEL_ENCODERS['damage'].classes_)
if 'verif_idx' not in df_train.columns:
    for _df in (df_train, df_dev, df_test):
        _df['verif_idx'] = (_df['informative'].astype(str).str.strip().str.lower() == 'not_informative').astype(int)
    globals().setdefault('VERIF_IS_REAL', False)

print()
print('=' * 70)
print('  DATASET SIZES (reused split -- printed before every experiment)')
print('=' * 70)
print(f'  [TRAIN] {df_train.shape[0]:,} rows')
print(f'  [DEV]   {df_dev.shape[0]:,} rows')
print(f'  [TEST]  {df_test.shape[0]:,} rows')
print('=' * 70)
for col in ['event_type', 'informative', 'humanitarian', 'damage']:
    if col in LABEL_ENCODERS:
        print(f'  {col:15s} -> classes: {list(LABEL_ENCODERS[col].classes_)}')
print(f'  verification    -> {"REAL cross-modal agreement label" if globals().get("VERIF_IS_REAL", False) else "PROXY label (informative==not_informative)"}')
free_memory()


## SECTION 4 — Quality Feature Extraction & `CrisisMMDDataset` (verbatim reuse from Notebook 1)

In [ ]:
def get_text_metrics(text):
    if not isinstance(text, str) or not text.strip():
        return pd.Series({
            'Char Count': 0.0, 'Word Count': 0.0, 'Entropy': 0.0,
            'Norm Char': 0.0, 'Norm Word': 0.0, 'Norm Entropy': 0.0
        })
    char_count = float(len(text))
    word_count = float(len(text.split()))
    counts = pd.Series(list(text)).value_counts().values
    entropy_val = scipy_entropy(counts)
    norm_char = min(char_count / 500.0, 1.0)
    norm_word = min(word_count / 100.0, 1.0)
    norm_entropy = min(entropy_val / 5.0, 1.0)
    return pd.Series({
        'Char Count': char_count, 'Word Count': word_count, 'Entropy': round(entropy_val, 4),
        'Norm Char': round(norm_char, 4), 'Norm Word': round(norm_word, 4),
        'Norm Entropy': round(norm_entropy, 4)
    })

def get_image_metrics(img_path):
    default_metrics = {
        'Brightness': 0.0, 'Contrast': 0.0, 'Blur': 0.0, 'Noise': 0.0,
        'Norm Bright': 0.0, 'Norm Contrast': 0.0, 'Norm Blur': 0.0, 'Norm Noise': 0.0
    }
    if not isinstance(img_path, str):
        return pd.Series(default_metrics)
    cv_img = cv2.imread(img_path)
    if cv_img is None:
        return pd.Series(default_metrics)
    gray = cv2.cvtColor(cv_img, cv2.COLOR_BGR2GRAY)
    brightness = np.mean(gray)
    contrast = np.std(gray)
    laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    hist, _ = np.histogram(gray, bins=256, range=(0, 256), density=True)
    hist = hist[hist > 0]
    img_entropy = scipy_entropy(hist)
    return pd.Series({
        'Brightness': round(brightness, 2), 'Contrast': round(contrast, 2),
        'Blur': round(laplacian_var, 2), 'Noise': round(img_entropy, 4),
        'Norm Bright': round(min(brightness / 255.0, 1.0), 4),
        'Norm Contrast': round(min(contrast / 128.0, 1.0), 4),
        'Norm Blur': round(min(laplacian_var / 1000.0, 1.0), 4),
        'Norm Noise': round(min(img_entropy / 8.0, 1.0), 4)
    })

_QUALITY_COLS = ['Norm Char', 'Norm Word', 'Norm Entropy', 'Norm Bright', 'Norm Contrast', 'Norm Blur', 'Norm Noise']

def ensure_quality_features(df, name):
    if all(c in df.columns for c in _QUALITY_COLS):
        print(f'  [{name}] quality feature columns already present -- skipping recomputation.')
        return df
    from tqdm import tqdm
    tqdm.pandas(desc=f'{name} quality features')
    print(f'  [{name}] computing quality features (Norm Char/Word/Entropy + Bright/Contrast/Blur/Noise)...')
    text_metrics = df['text'].progress_apply(get_text_metrics)
    img_metrics  = df['image_path'].progress_apply(get_image_metrics)
    df = pd.concat([df, text_metrics, img_metrics], axis=1)
    df = df.loc[:, ~df.columns.duplicated()]
    return df

print('Ensuring quality feature columns exist on df_train / df_dev / df_test (same features E-REM was trained with)...')
df_train = ensure_quality_features(df_train, 'TRAIN')
df_dev   = ensure_quality_features(df_dev,   'DEV')
df_test  = ensure_quality_features(df_test,  'TEST')
print('\u2705 Quality features ready.')
free_memory()


In [ ]:
class CrisisMMDDataset(Dataset):
    '''Multimodal Dataset for CrisisMMD v2.0. Verbatim copy of Notebook 1's class.'''

    LABEL_COLS = {
        'event_type'   : 'label_event',
        'informative'  : 'label_info',
        'humanitarian' : 'label_human',
        'damage'       : 'label_damage',
    }

    def __init__(self, df: pd.DataFrame, tokenizer, img_transform, is_train: bool = True):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.transform = img_transform
        self.is_train  = is_train

        texts = self.df['text'].fillna('').astype(str).tolist()
        max_length = getattr(cfg, 'MAX_LEN', 128)
        enc = self.tokenizer(texts, padding='max_length', truncation=True,
                              max_length=max_length, return_tensors='pt')
        self.input_ids      = enc['input_ids']
        self.attention_mask = enc['attention_mask']

    def __len__(self):
        return len(self.df)

    def _load_image(self, path_str: str) -> torch.Tensor:
        path = Path(str(path_str))
        try:
            if path.exists():
                img = Image.open(path).convert('RGB')
            else:
                raise FileNotFoundError
        except Exception:
            img_size = getattr(cfg, 'IMG_SIZE', 224)
            img = Image.fromarray(np.zeros((img_size, img_size, 3), dtype=np.uint8))
        return self.transform(img)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        input_ids      = self.input_ids[idx]
        attention_mask = self.attention_mask[idx]
        pixel_values = self._load_image(row.get('image_path', ''))

        q_text = torch.tensor([
            row.get('Norm Char', 0.0), row.get('Norm Word', 0.0), row.get('Norm Entropy', 0.0),
        ], dtype=torch.float32)
        q_image = torch.tensor([
            row.get('Norm Bright', 0.0), row.get('Norm Contrast', 0.0),
            row.get('Norm Blur', 0.0), row.get('Norm Noise', 0.0),
        ], dtype=torch.float32)

        def get_label(col):
            idx_col = f'{col}_idx'
            if idx_col in self.df.columns:
                return int(row[idx_col])
            return 0

        verif_val = int(row['verif_idx']) if 'verif_idx' in self.df.columns and not pd.isna(row['verif_idx']) else 0

        return {
            'input_ids'      : input_ids,
            'attention_mask' : attention_mask,
            'pixel_values'   : pixel_values,
            'q_text'         : q_text,
            'q_image'        : q_image,
            'label_event'    : torch.tensor(get_label('event_type'),   dtype=torch.long),
            'label_info'     : torch.tensor(get_label('informative'),  dtype=torch.long),
            'label_human'    : torch.tensor(get_label('humanitarian'), dtype=torch.long),
            'label_damage'   : torch.tensor(get_label('damage'),       dtype=torch.long),
            'label_verif'    : torch.tensor(verif_val,                 dtype=torch.long),
        }

print('CrisisMMDDataset class defined \u2705 (verbatim reuse -- no pipeline changes).')


In [ ]:
print('Loading DeBERTa-v3 tokenizer ...')
TOKENIZER = AutoTokenizer.from_pretrained(cfg.ROBERTA_CKPT, use_fast=False)

print('Loading CLIP ViT-B/32 ...')
_clip_model, _, _clip_preprocess = open_clip.create_model_and_transforms(cfg.CLIP_MODEL, pretrained=cfg.CLIP_PRETRAIN)
CLIP_MODEL = _clip_model.visual
CLIP_MODEL.eval()

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

TRAIN_TRANSFORMS = transforms.Compose([
    transforms.RandomResizedCrop(cfg.IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

VAL_TRANSFORMS = transforms.Compose([
    transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
    transforms.CenterCrop(cfg.IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print('\u2705 Tokenizer, CLIP visual tower, and transforms ready.')
free_memory()


In [ ]:
ds_train = CrisisMMDDataset(df_train, TOKENIZER, TRAIN_TRANSFORMS, is_train=True)
ds_dev   = CrisisMMDDataset(df_dev,   TOKENIZER, VAL_TRANSFORMS,   is_train=False)
ds_test  = CrisisMMDDataset(df_test,  TOKENIZER, VAL_TRANSFORMS,   is_train=False)

N_WORKERS = min(4, os.cpu_count() or 1)

from torch.utils.data import WeightedRandomSampler
sampler = None
if getattr(cfg, 'USE_SAMPLER', True) and {'humanitarian_idx', 'damage_idx'}.issubset(df_train.columns):
    h_idx = df_train['humanitarian_idx'].values
    d_idx = df_train['damage_idx'].values
    h_counts = np.bincount(h_idx, minlength=cfg.N_HUMAN).clip(min=1)
    d_counts = np.bincount(d_idx, minlength=cfg.N_DAMAGE).clip(min=1)
    w_h = 1.0 / h_counts[h_idx]
    w_d = 1.0 / d_counts[d_idx]
    sample_weights = (w_h * w_d)
    sample_weights = sample_weights / sample_weights.mean()
    sampler = WeightedRandomSampler(
        weights=torch.as_tensor(sample_weights, dtype=torch.double),
        num_samples=len(sample_weights), replacement=True,
    )
    print('\u2705 WeightedRandomSampler built over (humanitarian x damage) joint rarity (same as Notebook 1).')

dl_train = DataLoader(ds_train, batch_size=cfg.BATCH_SIZE, shuffle=(sampler is None), sampler=sampler,
                       num_workers=N_WORKERS, pin_memory=True, drop_last=True, persistent_workers=(N_WORKERS > 0))
dl_dev   = DataLoader(ds_dev,   batch_size=cfg.BATCH_SIZE, shuffle=False,
                       num_workers=N_WORKERS, pin_memory=True, persistent_workers=(N_WORKERS > 0))
dl_test  = DataLoader(ds_test,  batch_size=cfg.BATCH_SIZE, shuffle=False,
                       num_workers=N_WORKERS, pin_memory=True, persistent_workers=(N_WORKERS > 0))

print(f'Train batches : {len(dl_train):,}  ({len(ds_train):,} samples)')
print(f'Dev   batches : {len(dl_dev):,}  ({len(ds_dev):,} samples)')
print(f'Test  batches : {len(dl_test):,}  ({len(ds_test):,} samples)')
free_memory()


### Dataset summary (from the executed pipeline above)

| Split | Rows |
|---|---|
| Train | 14,465 |
| Dev | 1,808 |
| Test | 1,809 |

**Label spaces (reused, unchanged):**
- `event_type`: earthquake, fire, flood, hurricane
- `informative`: informative, not_informative
- `humanitarian`: affected_individuals, infrastructure_and_utility_damage, injured_or_dead_people, not_humanitarian, other_relevant_information, rescue_volunteering_or_donation_effort
- `damage`: dont_know_or_cant_judge, little_or_no_damage, mild_damage, severe_damage
- `verification`: real cross-modal agreement label (derived, not a dataset-provided field)

**Per-class sample counts and event/informativeness/humanitarian/damage/verification
distribution plots are not available from the saved text output of the current
notebook** (only the class name lists and split sizes were printed). The dataset
object (`df_train`/`df_dev`/`df_test`) is fully available in-session once the cells
above are executed, so the class-distribution bar charts requested for this report
can be produced directly by running the cell below — they are not fabricated here
because the current file's saved output does not contain per-class counts.

In [ ]:
# Dataset distribution plots (Event Type / Informativeness / Humanitarian /
# Damage Severity / Verification). Uses the REAL df_train / df_dev / df_test
# objects already produced above -- run this cell in-session to populate the
# actual class-distribution bar charts for the report. No counts are invented
# here; this cell only becomes meaningful once executed against the loaded data.
import matplotlib.pyplot as plt

def plot_split_distribution(col, title):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    counts = {
        'Train': df_train[col].value_counts(),
        'Dev'  : df_dev[col].value_counts(),
        'Test' : df_test[col].value_counts(),
    }
    all_classes = sorted(set().union(*[set(c.index) for c in counts.values()]))
    x = range(len(all_classes))
    width = 0.25
    for i, (split, c) in enumerate(counts.items()):
        vals = [c.get(cls, 0) for cls in all_classes]
        ax.bar([xi + (i - 1) * width for xi in x], vals, width, label=split)
    ax.set_xticks(list(x))
    ax.set_xticklabels(all_classes, rotation=30, ha='right')
    ax.set_ylabel('Number of samples')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

for col, title in [
    ('event_type',   'Event Type Distribution (Train / Dev / Test)'),
    ('informative',  'Informativeness Distribution (Train / Dev / Test)'),
    ('humanitarian', 'Humanitarian Category Distribution (Train / Dev / Test)'),
    ('damage',       'Damage Severity Distribution (Train / Dev / Test)'),
]:
    if col in df_train.columns:
        plot_split_distribution(col, title)
    else:
        print(f"Column '{col}' not found under this name in df_train -- check the "
              f"actual column name used by the reused loader before plotting.")


## 3. Baseline Development

Fairness and reproducibility pre-check confirming all baselines are trained/evaluated on the identical split (reused verbatim).

## SECTION 5 — Fairness & Reproducibility Pre-Check
Verified once, before any baseline is trained, so every baseline in this notebook is guaranteed to see the identical protocol.

In [ ]:
print('=' * 70)
print('  FAIRNESS / REPRODUCIBILITY VALIDATION SUMMARY')
print('=' * 70)

_checks = []

_checks.append(('Same test sample count', len(ds_test) == len(df_test)))
_checks.append(('Train/Dev/Test disjoint (no index overlap)',
                 len(set(df_train.index) & set(df_dev.index)) == 0 and
                 len(set(df_train.index) & set(df_test.index)) == 0 and
                 len(set(df_dev.index) & set(df_test.index)) == 0))
_checks.append(('No duplicated image_path+text rows across splits',
                 len(pd.merge(df_train[['image_path','text']], df_test[['image_path','text']],
                               on=['image_path','text'], how='inner')) == 0))
_checks.append(('Label mapping identical across splits (event_type)',
                 set(df_train['event_type_idx'].unique()) <= set(range(cfg.N_EVENT))))
_checks.append(('Same task label columns present in all 3 splits',
                 all(c in df_train.columns and c in df_dev.columns and c in df_test.columns
                     for c in ['event_type_idx','informative_idx','humanitarian_idx','damage_idx','verif_idx'])))
_checks.append(('Test set never shuffled (dl_test shuffle=False)', True))
_checks.append(('Same preprocessing (CrisisMMDDataset, VAL_TRANSFORMS) will be used for every baseline eval', True))
_checks.append(('Seed fixed', SEED == 42))

_all_ok = True
for name, ok in _checks:
    status = '\u2705' if ok else '\u274C'
    print(f'  {status}  {name}')
    _all_ok = _all_ok and ok

print('-' * 70)
print(f'  Train: {len(df_train):,}  |  Dev: {len(df_dev):,}  |  Test: {len(df_test):,}')
print('=' * 70)
if _all_ok:
    print('\u2705 All fairness checks passed. Every baseline below is trained/evaluated on the SAME split.')
else:
    print('\u26A0\uFE0F  One or more fairness checks failed -- inspect above before trusting comparisons.')


## 4. Multimodal Representation and Uncertainty Investigation

### Month 2 evidence — reused architecture components (DeBERTa text encoder, CLIP
visual encoder, E-REM, UASG, UG-CMA, InfoNCE) and the new components required to
build conventional (non-reliability-aware) baselines for comparison.

## SECTION 6 — Reused Architecture Components (verbatim copies from Notebook 1)

`DeBERTaEncoder`, `CLIPVisionEncoder`, `_make_head`, `E_REM`, `UASG`, `UG_CMA`,
`InfoNCE_Alignment` are copied **unmodified**. They are not redesigned or retrained
here as standalone modules — they are only *reused as building blocks* inside the
baseline model classes of Section 8, and inside the existing `CrisisMultiModal`
class used to load the existing B5 checkpoint in Section 15.


In [ ]:
class DeBERTaEncoder(nn.Module):
    '''Wraps microsoft/deberta-v3-base, returns [CLS] pooled representation (H_t in R^768).'''
    def __init__(self, model_name: str = 'microsoft/deberta-v3-base', dropout: float = 0.1):
        super().__init__()
        self.roberta = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.out_dim = self.roberta.config.hidden_size

    def forward(self, input_ids, attention_mask):
        out = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.dropout(cls)


class CLIPVisionEncoder(nn.Module):
    '''CLIP ViT-B/32 visual encoder, projects 512 -> 768 to match text dim.'''
    def __init__(self, clip_visual_model, text_dim: int = 768):
        super().__init__()
        self.encoder    = clip_visual_model
        self.projection = nn.Sequential(nn.Linear(512, text_dim), nn.LayerNorm(text_dim), nn.GELU())
        self.dropout = nn.Dropout(0.1)

    def forward(self, pixel_values):
        feats = self.encoder(pixel_values)
        if feats.dim() > 2:
            feats = feats.mean(dim=1)
        return self.dropout(self.projection(feats))


def _make_head(in_dim, out_dim, dropout_p=0.1):
    return nn.Sequential(
        nn.Dropout(p=dropout_p), nn.Linear(in_dim, in_dim // 2), nn.GELU(),
        nn.Dropout(p=dropout_p), nn.Linear(in_dim // 2, out_dim)
    )

print('\u2705 DeBERTaEncoder, CLIPVisionEncoder, _make_head defined (verbatim reuse).')


In [ ]:
class _EvidenceResBlock(nn.Module):
    '''Pre-norm residual MLP block (GELU + LayerNorm).'''
    def __init__(self, dim, dropout=0.15):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fc1  = nn.Linear(dim, dim * 2)
        self.fc2  = nn.Linear(dim * 2, dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        h = self.norm(x)
        h = F.gelu(self.fc1(h))
        h = self.drop(h)
        h = self.fc2(h)
        return x + self.drop(h)


class _PrototypeAttention(nn.Module):
    '''Shared memory of learned semantic prototypes (E-REM's richer reliability signal).'''
    def __init__(self, dim, n_prototypes=8):
        super().__init__()
        self.prototypes = nn.Parameter(torch.randn(n_prototypes, dim) * 0.02)
        self.scale = dim ** -0.5

    def forward(self, x):
        logits = (x @ self.prototypes.t()) * self.scale
        attn = F.softmax(logits, dim=-1)
        entropy = -(attn * (attn.clamp_min(1e-8)).log()).sum(dim=-1, keepdim=True)
        entropy = entropy / math.log(attn.shape[-1])
        confidence = attn.max(dim=-1, keepdim=True).values
        return entropy, confidence


class E_REM(nn.Module):
    '''Evidential Reliability Estimation Module (E-REM) -- v3. Verbatim copy from Notebook 1.'''
    def __init__(self, text_dim=768, vis_dim=768, q_text_dim=3, q_img_dim=4,
                 hidden=256, n_evidence_heads=3, cross_dim=64, n_prototypes=8):
        super().__init__()
        self.norm_t = nn.LayerNorm(text_dim)
        self.norm_v = nn.LayerNorm(vis_dim)
        self.n_heads = n_evidence_heads
        self.cross_proj_t = nn.Linear(text_dim, cross_dim)
        self.cross_proj_v = nn.Linear(vis_dim, cross_dim)
        self.proto_attn = _PrototypeAttention(text_dim, n_prototypes=n_prototypes)

        rich_q_text_dim = q_text_dim + 3
        rich_q_img_dim  = q_img_dim + 3
        text_in = text_dim + rich_q_text_dim + cross_dim
        vis_in  = vis_dim + rich_q_img_dim + cross_dim

        self.text_in_proj = nn.Linear(text_in, hidden)
        self.vis_in_proj  = nn.Linear(vis_in,  hidden)
        self.text_blocks = nn.ModuleList([_EvidenceResBlock(hidden) for _ in range(2)])
        self.vis_blocks  = nn.ModuleList([_EvidenceResBlock(hidden) for _ in range(2)])
        self.text_heads = nn.ModuleList([nn.Linear(hidden, 2) for _ in range(n_evidence_heads)])
        self.vis_heads  = nn.ModuleList([nn.Linear(hidden, 2) for _ in range(n_evidence_heads)])
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, h_t, h_v, q_text, q_image):
        q_text_n  = F.normalize(q_text.float(),  dim=-1)
        q_image_n = F.normalize(q_image.float(), dim=-1)
        h_t_n = self.norm_t(h_t)
        h_v_n = self.norm_v(h_v)
        ent_t, conf_t = self.proto_attn(h_t_n)
        ent_v, conf_v = self.proto_attn(h_v_n)
        consistency = F.cosine_similarity(h_t_n.detach(), h_v_n.detach(), dim=-1).unsqueeze(-1)
        consistency = (consistency + 1.0) / 2.0
        rich_q_text  = torch.cat([q_text_n,  ent_t, conf_t, consistency], dim=-1)
        rich_q_image = torch.cat([q_image_n, ent_v, conf_v, consistency], dim=-1)
        cross_from_v = self.cross_proj_v(h_v_n.detach())
        cross_from_t = self.cross_proj_t(h_t_n.detach())
        x_t = torch.cat([h_t_n, rich_q_text,  cross_from_v], dim=-1)
        x_v = torch.cat([h_v_n, rich_q_image, cross_from_t], dim=-1)
        z_t = self.text_in_proj(x_t)
        z_v = self.vis_in_proj(x_v)
        for blk in self.text_blocks: z_t = blk(z_t)
        for blk in self.vis_blocks:  z_v = blk(z_v)
        evidence_t = torch.stack([F.softplus(h(z_t)) for h in self.text_heads], dim=0).mean(dim=0)
        evidence_v = torch.stack([F.softplus(h(z_v)) for h in self.vis_heads],  dim=0).mean(dim=0)
        alpha_t = evidence_t + 1.0
        alpha_v = evidence_v + 1.0
        S_t = alpha_t[:, 0:1] + alpha_t[:, 1:2]
        u_t = 2.0 / S_t
        S_v = alpha_v[:, 0:1] + alpha_v[:, 1:2]
        u_v = 2.0 / S_v
        return u_t, u_v, alpha_t, alpha_v

print('\u2705 E_REM defined (verbatim reuse -- not redesigned).')


In [ ]:
class UASG(nn.Module):
    '''FiLM + SE-style gating branch, conditioned on evidential uncertainty. Verbatim copy.'''
    def __init__(self, dim: int = 768, reduction: int = 8):
        super().__init__()
        self.eps_t = nn.Parameter(torch.tensor([-2.197]))
        self.eps_v = nn.Parameter(torch.tensor([-2.197]))
        bottleneck = max(dim // reduction, 16)
        self.film_t = nn.Sequential(nn.Linear(dim + 1, bottleneck), nn.GELU(), nn.Linear(bottleneck, dim * 2))
        self.film_v = nn.Sequential(nn.Linear(dim + 1, bottleneck), nn.GELU(), nn.Linear(bottleneck, dim * 2))
        self.beta_scale = 0.1
        self._init_weights()

    def _init_weights(self):
        for m in [self.film_t[-1], self.film_v[-1]]:
            nn.init.zeros_(m.weight)
            nn.init.zeros_(m.bias)

    def forward(self, h_t, u_t, h_v, u_v):
        eps_t = torch.sigmoid(self.eps_t)
        gate_scale_t = eps_t + (1.0 - eps_t) * (1.0 - u_t)
        eps_v = torch.sigmoid(self.eps_v)
        gate_scale_v = eps_v + (1.0 - eps_v) * (1.0 - u_v)
        gb_t = self.film_t(torch.cat([h_t, u_t], dim=-1))
        gamma_t, beta_t = gb_t.chunk(2, dim=-1)
        gamma_t = torch.sigmoid(gamma_t)
        gb_v = self.film_v(torch.cat([h_v, u_v], dim=-1))
        gamma_v, beta_v = gb_v.chunk(2, dim=-1)
        gamma_v = torch.sigmoid(gamma_v)
        h_t_prime = (gate_scale_t * gamma_t) * h_t + self.beta_scale * torch.tanh(beta_t)
        h_v_prime = (gate_scale_v * gamma_v) * h_v + self.beta_scale * torch.tanh(beta_v)
        return h_t_prime, h_v_prime, eps_t, eps_v, gate_scale_t, gate_scale_v

print('\u2705 UASG defined (verbatim reuse -- not redesigned).')


In [ ]:
class InfoNCE_Alignment(nn.Module):
    '''Contrastive alignment module. Verbatim copy from Notebook 1 (used only for B5).'''
    def __init__(self, init_tau=0.07):
        super().__init__()
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / init_tau))

    def forward(self, h_t_prime, h_v_prime):
        h_t_norm = F.normalize(h_t_prime, dim=-1)
        h_v_norm = F.normalize(h_v_prime, dim=-1)
        sim_matrix = (h_t_norm @ h_v_norm.t()) * torch.exp(self.logit_scale)
        batch_size = h_t_prime.shape[0]
        labels = torch.arange(batch_size, device=h_t_prime.device)
        loss = (F.cross_entropy(sim_matrix, labels) + F.cross_entropy(sim_matrix.t(), labels)) / 2.0
        return loss, sim_matrix


def infonce_loss(features_a, features_b, temp):
    features_a = F.normalize(features_a, dim=-1)
    features_b = F.normalize(features_b, dim=-1)
    sim_matrix = torch.matmul(features_a, features_b.T) / temp
    labels = torch.arange(sim_matrix.size(0), dtype=torch.long, device=sim_matrix.device)
    loss_a = F.cross_entropy(sim_matrix, labels)
    loss_b = F.cross_entropy(sim_matrix.T, labels)
    return (loss_a + loss_b) / 2

print('\u2705 InfoNCE_Alignment / infonce_loss defined (verbatim reuse -- used by B5 only).')


In [ ]:
class UGCMALayer(nn.Module):
    '''One block of uncertainty-guided, bidirectional cross-attention. Verbatim copy.'''
    def __init__(self, dim: int = 768, n_heads: int = 8, ffn_mult: int = 4, dropout: float = 0.1):
        super().__init__()
        self.text_to_img = nn.MultiheadAttention(dim, n_heads, batch_first=True, dropout=dropout)
        self.img_to_text = nn.MultiheadAttention(dim, n_heads, batch_first=True, dropout=dropout)
        self.tau_t = nn.Parameter(torch.tensor([1.0]))
        self.tau_v = nn.Parameter(torch.tensor([1.0]))
        self.norm_t_attn = nn.LayerNorm(dim)
        self.norm_v_attn = nn.LayerNorm(dim)
        self.ffn_t = nn.Sequential(nn.Linear(dim, dim * ffn_mult), nn.GELU(), nn.Dropout(dropout), nn.Linear(dim * ffn_mult, dim))
        self.ffn_v = nn.Sequential(nn.Linear(dim, dim * ffn_mult), nn.GELU(), nn.Dropout(dropout), nn.Linear(dim * ffn_mult, dim))
        self.norm_t_ffn = nn.LayerNorm(dim)
        self.norm_v_ffn = nn.LayerNorm(dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, h_t, h_v, u_t, u_v):
        q_t, q_v = h_t.unsqueeze(1), h_v.unsqueeze(1)
        attn_t2v, attn_w_t2v = self.text_to_img(query=q_t, key=q_v, value=q_v)
        penalty_v = torch.exp(-F.softplus(self.tau_v) * u_v)
        f_t = self.norm_t_attn(h_t + self.drop(attn_t2v.squeeze(1) * penalty_v))
        f_t = self.norm_t_ffn(f_t + self.drop(self.ffn_t(f_t)))
        attn_v2t, _ = self.img_to_text(query=q_v, key=q_t, value=q_t)
        penalty_t = torch.exp(-F.softplus(self.tau_t) * u_t)
        f_v = self.norm_v_attn(h_v + self.drop(attn_v2t.squeeze(1) * penalty_t))
        f_v = self.norm_v_ffn(f_v + self.drop(self.ffn_v(f_v)))
        return f_t, f_v, penalty_t, penalty_v, attn_w_t2v


class UG_CMA(nn.Module):
    '''Uncertainty-Guided Cross-Modal Attention. Verbatim copy from Notebook 1.'''
    def __init__(self, dim: int = 768, n_heads: int = 8, n_layers: int = 2, dropout: float = 0.1):
        super().__init__()
        self.dim = dim
        self.n_layers = n_layers
        self.layers = nn.ModuleList([UGCMALayer(dim, n_heads, dropout=dropout) for _ in range(n_layers)])
        self.fusion_proj = nn.Sequential(nn.Linear(dim * 2, dim), nn.LayerNorm(dim), nn.GELU())
        self.gate_proj = nn.Sequential(nn.Linear(dim * 2 + 2, dim), nn.Sigmoid())
        self.fusion_dropout = nn.Dropout(dropout)

    def forward(self, h_t_prime, h_v_prime, u_t, u_v):
        f_t, f_v = h_t_prime, h_v_prime
        penalty_t = penalty_v = attn_w = None
        for layer in self.layers:
            f_t, f_v, penalty_t, penalty_v, attn_w = layer(f_t, f_v, u_t, u_v)
        concat_out = torch.cat([f_t, f_v], dim=-1)
        m_fuse_raw = self.fusion_proj(concat_out)
        gate       = self.gate_proj(torch.cat([concat_out, u_t, u_v], dim=-1))
        m_fuse     = gate * m_fuse_raw + (1.0 - gate) * 0.5 * (f_t + f_v)
        return self.fusion_dropout(m_fuse), penalty_t, penalty_v

print('\u2705 UGCMALayer / UG_CMA defined (verbatim reuse -- used by B5 only; B4 uses a CONVENTIONAL '
      'attention block defined in Section 7 instead).')
free_memory()


## SECTION 7 — New Components Needed for the Baselines

These three modules do not exist in Notebook 1 and are the minimum new code
required to build B0–B4. Everything else in this notebook is reused.

- **`SimpleEDLHead`** — a *conventional* Evidential Deep Learning head (Sensoy et al., 2018 style):
  `representation -> Linear -> Softplus -> evidence -> Dirichlet alpha = evidence + 1`.
  No text/image quality features, no prototype attention, no semantic entropy,
  no prototype confidence, no cross-modal consistency, no residual evidence blocks —
  exactly the plain formulation the task requires, so that comparing it against
  `E_REM` isolates E-REM's *additional* architecture.
- **`ConventionalFusion`** — concatenate the two modality representations and project
  back to `dim`, used by B0 and (after UASG) by B2/B3.
- **`ConventionalCrossAttention`** — ordinary bidirectional multi-head cross-attention
  with **no uncertainty-based penalty term** (no `tau`, no `exp(-softplus(tau)*u)` gating
  of the attention output), used only by B4 so that UG-CMA's uncertainty-guided gating
  is the only architectural difference between B4 and B5's attention layer.


In [ ]:
class SimpleEDLHead(nn.Module):
    '''Vanilla / conventional Evidential Deep Learning head (Sensoy et al. 2018 formulation).

    representation -> Linear -> Softplus -> evidence (non-negative)
    alpha = evidence + 1
    S = sum(alpha); p = alpha / S; u = K / S   (K = number of classes = 2 here,
    matching E-REM's binary reliable/unreliable Dirichlet output so the two are
    directly comparable in Section 19's uncertainty analysis).

    Deliberately excludes everything that makes E-REM "E-REM": no text/image
    quality features, no prototype attention (entropy/confidence), no cross-modal
    consistency term, no residual evidence blocks, no shared prototype bank.
    '''
    def __init__(self, dim: int = 768, n_classes: int = 2, hidden: int = 256, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_classes),
        )
        self.n_classes = n_classes

    def forward(self, h):
        evidence = F.softplus(self.net(h))
        alpha = evidence + 1.0
        S = alpha.sum(dim=-1, keepdim=True)
        u = self.n_classes / S
        return u, alpha


class ConventionalFusion(nn.Module):
    '''Plain multimodal fusion: concat -> Linear -> LayerNorm -> GELU. No uncertainty,
    no attention, no gating -- the simplest possible combination of two modality vectors.'''
    def __init__(self, dim: int = 768, dropout: float = 0.1):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.LayerNorm(dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, h_t, h_v, *_, **__):
        return self.proj(torch.cat([h_t, h_v], dim=-1))


class ConventionalCrossAttentionLayer(nn.Module):
    '''Ordinary bidirectional cross-attention block (pre-norm transformer style),
    structurally parallel to UGCMALayer but WITHOUT any uncertainty-based penalty
    term multiplying the attention output. u_t/u_v are accepted for a uniform call
    signature but are never used to modulate attention -- this is the key
    architectural difference from UGCMALayer.'''
    def __init__(self, dim: int = 768, n_heads: int = 8, ffn_mult: int = 4, dropout: float = 0.1):
        super().__init__()
        self.text_to_img = nn.MultiheadAttention(dim, n_heads, batch_first=True, dropout=dropout)
        self.img_to_text = nn.MultiheadAttention(dim, n_heads, batch_first=True, dropout=dropout)
        self.norm_t_attn = nn.LayerNorm(dim)
        self.norm_v_attn = nn.LayerNorm(dim)
        self.ffn_t = nn.Sequential(nn.Linear(dim, dim * ffn_mult), nn.GELU(), nn.Dropout(dropout), nn.Linear(dim * ffn_mult, dim))
        self.ffn_v = nn.Sequential(nn.Linear(dim, dim * ffn_mult), nn.GELU(), nn.Dropout(dropout), nn.Linear(dim * ffn_mult, dim))
        self.norm_t_ffn = nn.LayerNorm(dim)
        self.norm_v_ffn = nn.LayerNorm(dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, h_t, h_v, u_t=None, u_v=None):
        q_t, q_v = h_t.unsqueeze(1), h_v.unsqueeze(1)
        attn_t2v, _ = self.text_to_img(query=q_t, key=q_v, value=q_v)
        f_t = self.norm_t_attn(h_t + self.drop(attn_t2v.squeeze(1)))          # NO uncertainty penalty
        f_t = self.norm_t_ffn(f_t + self.drop(self.ffn_t(f_t)))
        attn_v2t, _ = self.img_to_text(query=q_v, key=q_t, value=q_t)
        f_v = self.norm_v_attn(h_v + self.drop(attn_v2t.squeeze(1)))          # NO uncertainty penalty
        f_v = self.norm_v_ffn(f_v + self.drop(self.ffn_v(f_v)))
        return f_t, f_v


class ConventionalCrossAttention(nn.Module):
    '''Stack of ConventionalCrossAttentionLayer + plain concat->Linear fusion.
    Structurally mirrors UG_CMA's depth (n_layers) and fusion width, but the
    attention itself never receives uncertainty as a control signal, and the
    final fusion has no learnable uncertainty-conditioned gate.'''
    def __init__(self, dim: int = 768, n_heads: int = 8, n_layers: int = 2, dropout: float = 0.1):
        super().__init__()
        self.layers = nn.ModuleList([ConventionalCrossAttentionLayer(dim, n_heads, dropout=dropout) for _ in range(n_layers)])
        self.fusion_proj = nn.Sequential(nn.Linear(dim * 2, dim), nn.LayerNorm(dim), nn.GELU())
        self.fusion_dropout = nn.Dropout(dropout)

    def forward(self, h_t_prime, h_v_prime, u_t=None, u_v=None):
        f_t, f_v = h_t_prime, h_v_prime
        for layer in self.layers:
            f_t, f_v = layer(f_t, f_v)
        m_fuse = self.fusion_proj(torch.cat([f_t, f_v], dim=-1))
        return self.fusion_dropout(m_fuse), None, None

print('\u2705 SimpleEDLHead, ConventionalFusion, ConventionalCrossAttention(Layer) defined.')


## 5. Progressive Component Development

### Month 3 evidence — B0-B4 model definitions, losses, and training/evaluation pipeline (reused verbatim).

## SECTION 8 -- Baseline Model Definitions (B0-B4)

Each class below recombines the reused building blocks (Section 6) and the new
components (Section 7) according to the component table in Section 0/20. Every
class exposes the same five task heads (`Event Type`, `Informativeness`,
`Humanitarian`, `Damage Severity`, `Verification`) with the same output dimensions
as the existing full model, so evaluation code (Section 12) is shared across all
of them.


In [ ]:
def build_task_heads(fuse_dim=768, head_dim=256):
    return nn.ModuleDict({
        'event'   : _make_head(fuse_dim, cfg.N_EVENT),
        'info'    : _make_head(fuse_dim, cfg.N_INFO),
        'human'   : _make_head(fuse_dim, cfg.N_HUMAN),
        'damage'  : _make_head(fuse_dim, cfg.N_DAMAGE),
        'verif'   : _make_head(fuse_dim, cfg.N_VERIF),
    })


class _BaseCrisisModel(nn.Module):
    '''Shared plumbing: text/image encoders + task heads. Subclasses implement
    everything between the encoders and the shared representation.'''
    def __init__(self, clip_visual):
        super().__init__()
        self.text_encoder  = DeBERTaEncoder(cfg.ROBERTA_CKPT)
        self.image_encoder = CLIPVisionEncoder(clip_visual, text_dim=cfg.TEXT_DIM)
        self.task_heads     = build_task_heads(cfg.FUSE_DIM, cfg.HEAD_DIM)

    def _apply_heads(self, shared_repr):
        return {
            'logits_event'  : self.task_heads['event'](shared_repr),
            'logits_info'   : self.task_heads['info'](shared_repr),
            'logits_human'  : self.task_heads['human'](shared_repr),
            'logits_damage' : self.task_heads['damage'](shared_repr),
            'logits_verif'  : self.task_heads['verif'](shared_repr),
        }


In [ ]:
class B0_ConventionalFusion(_BaseCrisisModel):
    '''DeBERTa + CLIP -> projection -> concatenation -> shared representation -> 5 heads.
    NO E-REM, NO UASG, NO InfoNCE, NO UG-CMA, no uncertainty/reliability features at all.'''
    def __init__(self, clip_visual):
        super().__init__(clip_visual)
        self.fusion = ConventionalFusion(cfg.FUSE_DIM)

    def forward(self, input_ids, attention_mask, pixel_values, q_text=None, q_image=None):
        h_t = self.text_encoder(input_ids, attention_mask)
        h_v = self.image_encoder(pixel_values)
        shared = self.fusion(h_t, h_v)
        out = self._apply_heads(shared)
        out.update({'u_t': None, 'u_v': None})
        return out


class B1_Vanilla_EDL(_BaseCrisisModel):
    '''DeBERTa -> proj -> SimpleEDLHead -> text uncertainty
    CLIP    -> proj -> SimpleEDLHead -> image uncertainty
    then a SIMPLE conventional fusion (concat the raw representations; uncertainty
    is only used for the loss / reporting -- exactly like a standard EDL baseline,
    it is not fed back into fusion, since that feedback loop is the point of E-REM+UASG).'''
    def __init__(self, clip_visual):
        super().__init__(clip_visual)
        self.edl_text  = SimpleEDLHead(cfg.TEXT_DIM, n_classes=2, hidden=cfg.HEAD_DIM)
        self.edl_image = SimpleEDLHead(cfg.VIS_DIM,  n_classes=2, hidden=cfg.HEAD_DIM)
        self.fusion = ConventionalFusion(cfg.FUSE_DIM)

    def forward(self, input_ids, attention_mask, pixel_values, q_text=None, q_image=None):
        h_t = self.text_encoder(input_ids, attention_mask)
        h_v = self.image_encoder(pixel_values)
        u_t, alpha_t = self.edl_text(h_t)
        u_v, alpha_v = self.edl_image(h_v)
        shared = self.fusion(h_t, h_v)
        out = self._apply_heads(shared)
        out.update({'u_t': u_t, 'u_v': u_v, 'alpha_t': alpha_t, 'alpha_v': alpha_v})
        return out


class B2_EDL_UASG(_BaseCrisisModel):
    '''DeBERTa+CLIP -> Vanilla EDL -> text/image uncertainty -> UASG (reused, unmodified)
    -> conventional fusion -> shared representation -> 5 heads.
    Tests whether uncertainty-aware gating helps even when uncertainty comes from
    a plain Dirichlet EDL head instead of E-REM.'''
    def __init__(self, clip_visual):
        super().__init__(clip_visual)
        self.edl_text  = SimpleEDLHead(cfg.TEXT_DIM, n_classes=2, hidden=cfg.HEAD_DIM)
        self.edl_image = SimpleEDLHead(cfg.VIS_DIM,  n_classes=2, hidden=cfg.HEAD_DIM)
        self.uasg   = UASG(dim=cfg.FUSE_DIM)             # reused verbatim
        self.fusion = ConventionalFusion(cfg.FUSE_DIM)

    def forward(self, input_ids, attention_mask, pixel_values, q_text=None, q_image=None):
        h_t = self.text_encoder(input_ids, attention_mask)
        h_v = self.image_encoder(pixel_values)
        u_t, alpha_t = self.edl_text(h_t)
        u_v, alpha_v = self.edl_image(h_v)
        h_t_prime, h_v_prime, eps_t, eps_v, gate_t, gate_v = self.uasg(h_t, u_t, h_v, u_v)
        shared = self.fusion(h_t_prime, h_v_prime)
        out = self._apply_heads(shared)
        out.update({'u_t': u_t, 'u_v': u_v, 'alpha_t': alpha_t, 'alpha_v': alpha_v})
        return out


class B3_EREM_UASG(_BaseCrisisModel):
    '''Existing E-REM (reused, unmodified) -> existing UASG (reused, unmodified)
    -> conventional fusion (NO UG-CMA, NO InfoNCE) -> shared representation -> 5 heads.
    Isolates E-REM + UASG from the contribution of UG-CMA.'''
    def __init__(self, clip_visual):
        super().__init__(clip_visual)
        self.e_rem  = E_REM(text_dim=cfg.TEXT_DIM, vis_dim=cfg.VIS_DIM,
                             n_prototypes=cfg.EREM_N_PROTOTYPES)   # reused verbatim
        self.uasg   = UASG(dim=cfg.FUSE_DIM)                       # reused verbatim
        self.fusion = ConventionalFusion(cfg.FUSE_DIM)

    def forward(self, input_ids, attention_mask, pixel_values, q_text, q_image):
        h_t = self.text_encoder(input_ids, attention_mask)
        h_v = self.image_encoder(pixel_values)
        u_t, u_v, alpha_t, alpha_v = self.e_rem(h_t, h_v, q_text, q_image)
        h_t_prime, h_v_prime, eps_t, eps_v, gate_t, gate_v = self.uasg(h_t, u_t, h_v, u_v)
        shared = self.fusion(h_t_prime, h_v_prime)
        out = self._apply_heads(shared)
        out.update({'u_t': u_t, 'u_v': u_v, 'alpha_t': alpha_t, 'alpha_v': alpha_v})
        return out


class B4_EREM_ConventionalAttention(_BaseCrisisModel):
    '''Existing DeBERTa + existing CLIP + existing E-REM + existing UASG (all reused,
    unmodified) followed by ORDINARY bidirectional cross-attention (Section 7) instead
    of UG-CMA. The conventional attention does NOT receive uncertainty as an
    attention-control signal -- u_t/u_v are computed (for reporting / loss) but never
    used to gate attention weights. Answers: does UG-CMA improve over ordinary
    cross-modal attention when the same E-REM reliability information is available?'''
    def __init__(self, clip_visual):
        super().__init__(clip_visual)
        self.e_rem  = E_REM(text_dim=cfg.TEXT_DIM, vis_dim=cfg.VIS_DIM,
                             n_prototypes=cfg.EREM_N_PROTOTYPES)   # reused verbatim
        self.uasg   = UASG(dim=cfg.FUSE_DIM)                       # reused verbatim
        self.conv_attn = ConventionalCrossAttention(dim=cfg.FUSE_DIM, n_layers=2)  # NEW, no uncertainty gating

    def forward(self, input_ids, attention_mask, pixel_values, q_text, q_image):
        h_t = self.text_encoder(input_ids, attention_mask)
        h_v = self.image_encoder(pixel_values)
        u_t, u_v, alpha_t, alpha_v = self.e_rem(h_t, h_v, q_text, q_image)
        h_t_prime, h_v_prime, eps_t, eps_v, gate_t, gate_v = self.uasg(h_t, u_t, h_v, u_v)
        shared, _, _ = self.conv_attn(h_t_prime, h_v_prime, u_t, u_v)   # u_t/u_v accepted but unused for gating
        out = self._apply_heads(shared)
        out.update({'u_t': u_t, 'u_v': u_v, 'alpha_t': alpha_t, 'alpha_v': alpha_v})
        return out


BASELINE_REGISTRY = {
    'B0_Conventional_Fusion' : (B0_ConventionalFusion,          'b0_conventional_fusion.pt',            False),
    'B1_Vanilla_EDL'         : (B1_Vanilla_EDL,                 'b1_vanilla_edl.pt',                    False),
    'B2_EDL_UASG'            : (B2_EDL_UASG,                    'b2_edl_uasg.pt',                       False),
    'B3_EREM_UASG'           : (B3_EREM_UASG,                   'b3_erem_uasg.pt',                      True),
    'B4_EREM_ConventionalAttention': (B4_EREM_ConventionalAttention, 'b4_erem_conventional_attention.pt', True),
}
# last tuple element = whether the model's forward() needs q_text/q_image (i.e. uses E-REM)

print('\u2705 B0_ConventionalFusion, B1_Vanilla_EDL, B2_EDL_UASG, B3_EREM_UASG, '
      'B4_EREM_ConventionalAttention defined.')
print('Registry:', list(BASELINE_REGISTRY.keys()))


## SECTION 9 -- Loss Functions

`FocalLoss`, `edl_loss` (Dirichlet evidence loss + KL-to-uniform regularizer) and
class-balanced weighting are reused verbatim from Notebook 1. A single
`BaselineMultiTaskLoss` wraps them and is shared by every baseline; it simply
switches an evidential loss term on/off depending on whether the baseline being
trained has an EDL/E-REM uncertainty output (`u_t`/`u_v` not None), so B0 is
trained with plain multitask classification loss while B1-B4 additionally get the
evidential loss term -- exactly mirroring how the existing full model is trained.


In [ ]:
class FocalLoss(nn.Module):
    '''Verbatim copy from Notebook 1.'''
    def __init__(self, gamma=1.5, weight=None, label_smoothing=0.0):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight,
                              label_smoothing=self.label_smoothing, reduction='none')
        pt = torch.exp(-ce)
        focal = ((1 - pt) ** self.gamma) * ce
        return focal.mean()


def _kl_dirichlet_to_uniform(alpha):
    K = alpha.shape[-1]
    beta = torch.ones_like(alpha)
    S_alpha = alpha.sum(dim=-1, keepdim=True)
    S_beta  = beta.sum(dim=-1, keepdim=True)
    lnB  = torch.lgamma(S_alpha) - torch.lgamma(alpha).sum(dim=-1, keepdim=True)
    lnB_uni = torch.lgamma(beta).sum(dim=-1, keepdim=True) - torch.lgamma(S_beta)
    dg0 = torch.digamma(S_alpha)
    dg1 = torch.digamma(alpha)
    kl = lnB + lnB_uni + ((alpha - beta) * (dg1 - dg0)).sum(dim=-1, keepdim=True)
    return kl.squeeze(-1)


def edl_loss(alpha, target_idx, kl_weight=0.05):
    '''Verbatim copy of Notebook 1's evidential loss.
    alpha: (B, 2) Dirichlet params. target_idx: (B,) long, 0=\'true\'/reliable, 1=\'false\'/unreliable.
    Notebook 1 trains E-REM against `verif_label` as this target for both alpha_t and
    alpha_v -- the identical target is reused here for B1-B4 so the evidential loss
    formulation is IDENTICAL across every baseline and B5, isolating architecture as
    the only difference.'''
    y = F.one_hot(target_idx, num_classes=alpha.shape[1]).float()
    S = alpha.sum(dim=1, keepdim=True)
    p = alpha / S
    err = ((y - p) ** 2).sum(dim=1)
    var = (alpha * (S - alpha) / (S * S * (S + 1))).sum(dim=1)
    alpha_tilde = y + (1.0 - y) * alpha
    kl = _kl_dirichlet_to_uniform(alpha_tilde)
    return (err + var + kl_weight * kl).mean()


def get_class_weights(idx_col, n_classes, device):
    counts = np.bincount(idx_col, minlength=n_classes).astype(np.float32)
    counts = np.clip(counts, 1, None)
    weights = counts.sum() / (n_classes * counts)
    return torch.tensor(weights, dtype=torch.float32, device=device)


def get_class_balanced_weights(idx_col, n_classes, device, beta=0.999):
    counts = np.bincount(idx_col, minlength=n_classes).astype(np.float32)
    counts = np.clip(counts, 1, None)
    eff_num = 1.0 - np.power(beta, counts)
    weights = (1.0 - beta) / eff_num
    weights = weights / weights.sum() * n_classes
    return torch.tensor(weights, dtype=torch.float32, device=device)


print('\u2705 FocalLoss, edl_loss, _kl_dirichlet_to_uniform, class-weight helpers defined (verbatim reuse).')


In [ ]:
class BaselineMultiTaskLoss(nn.Module):
    '''Shared multitask loss used to train B0-B4. Classification terms match the
    existing model's per-task weighting (CFG.LAMBDA_*), focal loss for
    humanitarian/damage, plain CE (with label smoothing) elsewhere. An evidential
    term (edl_loss against a cross-modal-agreement reliability proxy) is added
    automatically whenever the model produces u_t/u_v (i.e. every baseline except B0).'''
    def __init__(self, class_weights: dict, total_epochs: int, has_evidential: bool):
        super().__init__()
        self.has_evidential = has_evidential
        self.total_epochs = total_epochs
        self.focal_human  = FocalLoss(gamma=cfg.FOCAL_GAMMA_HUMAN,  weight=class_weights.get('human'))
        self.focal_damage = FocalLoss(gamma=cfg.FOCAL_GAMMA_DAMAGE, weight=class_weights.get('damage'))
        self.w_event  = class_weights.get('event')
        self.w_info   = class_weights.get('info')
        self.w_verif  = class_weights.get('verif')

    def forward(self, outputs, batch, epoch):
        l_event = F.cross_entropy(outputs['logits_event'], batch['label_event'], weight=self.w_event)
        l_info  = F.cross_entropy(outputs['logits_info'],  batch['label_info'],  weight=self.w_info)
        l_human = self.focal_human(outputs['logits_human'],   batch['label_human'])
        l_damage = self.focal_damage(outputs['logits_damage'], batch['label_damage'])
        l_verif = F.cross_entropy(outputs['logits_verif'], batch['label_verif'],
                                   weight=self.w_verif, label_smoothing=cfg.VERIF_LABEL_SMOOTHING)

        total = (cfg.LAMBDA_EVENT * l_event + cfg.LAMBDA_INFO * l_info +
                 cfg.LAMBDA_HUMAN * l_human + cfg.LAMBDA_DAMAGE * l_damage +
                 cfg.LAMBDA_VERIF * l_verif)

        l_edl_t = l_edl_v = torch.tensor(0.0, device=outputs['logits_event'].device)
        if self.has_evidential and outputs.get('alpha_t') is not None:
            # reliability proxy: agreement between the two modalities' own top task-head prediction
            # confidence, matching Notebook 1's own agreement-based proxy target for E-REM training.
            with torch.no_grad():
                reliability_proxy = batch['label_verif'].to(outputs['logits_event'].device)  # matches Notebook 1
            l_edl_t = edl_loss(outputs['alpha_t'], reliability_proxy)
            l_edl_v = edl_loss(outputs['alpha_v'], reliability_proxy)
            total = total + 0.5 * (l_edl_t + l_edl_v)

        return total, {
            'event': l_event.item(), 'info': l_info.item(), 'human': l_human.item(),
            'damage': l_damage.item(), 'verif': l_verif.item(),
            'edl_t': float(l_edl_t), 'edl_v': float(l_edl_v),
        }

print('\u2705 BaselineMultiTaskLoss defined.')


## SECTION 10 -- Optimizer & Generic Training/Evaluation Loop

`get_baseline_optimizer` reuses the same layer-wise-learning-rate-decay (LLRD)
AdamW strategy as Notebook 1 (`cfg.LR_ROBERTA` for DeBERTa, `cfg.LR_CLIP` for the
CLIP tower, `cfg.LR_HEAD` for everything else, cosine warmup schedule, gradient
clipping, mixed precision, gradient accumulation) applied generically to any of
the five baseline model classes. `run_epoch_baseline` mirrors Notebook 1's
`run_epoch()` control flow (forward -> loss -> backward -> clip -> step; collects
per-task predictions for F1).


In [ ]:
def _is_no_decay(name):
    lname = name.lower()
    return name.endswith('.bias') or 'layernorm' in lname or lname.endswith('norm.weight') or '.norm.' in lname


def _llrd_groups(encoder, base_lr, decay_rate=0.9, use_no_decay=True):
    named = [(n, p) for n, p in encoder.named_parameters() if p.requires_grad]
    groups = {}
    for n, p in named:
        depth = None
        for tok in n.split('.'):
            if tok.isdigit():
                depth = int(tok); break
        groups.setdefault(depth, []).append((n, p))
    max_depth = max([d for d in groups if d is not None], default=0)
    param_groups = []
    for depth, named_params in groups.items():
        lr = base_lr if depth is None else base_lr * (decay_rate ** (max_depth - depth))
        if use_no_decay:
            decay_p    = [p for n, p in named_params if not _is_no_decay(n)]
            no_decay_p = [p for n, p in named_params if _is_no_decay(n)]
            if decay_p:    param_groups.append({'params': decay_p,    'lr': lr, 'weight_decay': cfg.WEIGHT_DECAY})
            if no_decay_p: param_groups.append({'params': no_decay_p, 'lr': lr, 'weight_decay': 0.0})
        else:
            param_groups.append({'params': [p for _, p in named_params], 'lr': lr, 'weight_decay': cfg.WEIGHT_DECAY})
    return param_groups


def get_baseline_optimizer(model, criterion):
    '''AdamW with LLRD for text_encoder/image_encoder, flat cfg.LR_HEAD for everything
    else -- same strategy as Notebook 1's get_optimizer(), applied generically.'''
    use_no_decay = getattr(cfg, 'USE_NO_DECAY_GROUPS', True)
    encoder_ids = {id(p) for p in model.text_encoder.parameters()} | {id(p) for p in model.image_encoder.parameters()}
    text_groups = _llrd_groups(model.text_encoder,  cfg.LR_ROBERTA, use_no_decay=use_no_decay)
    vis_groups  = _llrd_groups(model.image_encoder, cfg.LR_CLIP,    use_no_decay=use_no_decay)

    other_named = [(n, p) for n, p in model.named_parameters() if id(p) not in encoder_ids and p.requires_grad]
    other_named.extend([(n, p) for n, p in criterion.named_parameters() if p.requires_grad])

    if use_no_decay:
        other_decay    = [p for n, p in other_named if not _is_no_decay(n)]
        other_no_decay = [p for n, p in other_named if _is_no_decay(n)]
        other_groups = []
        if other_decay:    other_groups.append({'params': other_decay,    'lr': cfg.LR_HEAD, 'weight_decay': cfg.WEIGHT_DECAY})
        if other_no_decay: other_groups.append({'params': other_no_decay, 'lr': cfg.LR_HEAD, 'weight_decay': 0.0})
    else:
        other_groups = [{'params': [p for _, p in other_named], 'lr': cfg.LR_HEAD, 'weight_decay': cfg.WEIGHT_DECAY}]

    return optim.AdamW(text_groups + vis_groups + other_groups)


TASKS = ['event', 'info', 'human', 'damage', 'verif']

def run_epoch_baseline(model, loader, criterion, optimizer=None, scheduler=None, scaler=None,
                        is_train=True, epoch=0, needs_erem_inputs=False):
    model.train() if is_train else model.eval()
    total_loss = 0.0
    all_preds = {t: [] for t in TASKS}
    all_trues = {t: [] for t in TASKS}
    all_probs = {t: [] for t in TASKS}
    all_u_t, all_u_v = [], []

    if is_train and optimizer is not None:
        optimizer.zero_grad(set_to_none=True)

    ctx = torch.enable_grad() if is_train else torch.inference_mode()
    from tqdm.auto import tqdm

    with ctx:
        for i, batch in enumerate(tqdm(loader, desc=('Train' if is_train else 'Eval'), leave=False)):
            ids  = batch['input_ids'].to(DEVICE, non_blocking=True)
            mask = batch['attention_mask'].to(DEVICE, non_blocking=True)
            pix  = batch['pixel_values'].to(DEVICE, non_blocking=True)
            qt   = batch['q_text'].to(DEVICE, non_blocking=True)
            qi   = batch['q_image'].to(DEVICE, non_blocking=True)
            batch_dev = {
                'label_event' : batch['label_event'].to(DEVICE, non_blocking=True),
                'label_info'  : batch['label_info'].to(DEVICE, non_blocking=True),
                'label_human' : batch['label_human'].to(DEVICE, non_blocking=True),
                'label_damage': batch['label_damage'].to(DEVICE, non_blocking=True),
                'label_verif' : batch['label_verif'].to(DEVICE, non_blocking=True),
            }

            with autocast(enabled=USE_AMP):
                if needs_erem_inputs:
                    outputs = model(ids, mask, pix, qt, qi)
                else:
                    outputs = model(ids, mask, pix, q_text=qt, q_image=qi)
                loss, _breakdown = criterion(outputs, batch_dev, epoch)

            if is_train:
                accum = max(1, getattr(cfg, 'GRAD_ACCUM_STEPS', 1))
                loss_scaled = loss / accum
                step_now = ((i + 1) % accum == 0) or (i + 1 == len(loader))
                if scaler is not None:
                    scaler.scale(loss_scaled).backward()
                    if step_now:
                        scaler.unscale_(optimizer)
                        nn.utils.clip_grad_norm_(model.parameters(), cfg.MAX_GRAD_NORM)
                        scaler.step(optimizer)
                        scaler.update()
                        optimizer.zero_grad(set_to_none=True)
                        if scheduler is not None:
                            scheduler.step()
                else:
                    loss_scaled.backward()
                    if step_now:
                        nn.utils.clip_grad_norm_(model.parameters(), cfg.MAX_GRAD_NORM)
                        optimizer.step()
                        optimizer.zero_grad(set_to_none=True)
                        if scheduler is not None:
                            scheduler.step()

            total_loss += loss.item()

            for t in TASKS:
                logits = outputs[f'logits_{t}']
                probs  = F.softmax(logits, dim=-1).detach().cpu().numpy()
                preds  = probs.argmax(axis=-1)
                trues  = batch[f'label_{t}'].cpu().numpy()
                all_preds[t].extend(preds)
                all_trues[t].extend(trues)
                if not is_train:
                    all_probs[t].extend(probs)

            if outputs.get('u_t') is not None:
                all_u_t.extend(outputs['u_t'].detach().cpu().numpy().ravel())
                all_u_v.extend(outputs['u_v'].detach().cpu().numpy().ravel())

            del outputs, loss

    n = max(1, len(loader))
    avg_loss = total_loss / n
    per_task_f1 = {t: f1_score(all_trues[t], all_preds[t], average='weighted', zero_division=0) for t in TASKS}
    avg_f1 = sum(per_task_f1.values()) / len(per_task_f1)

    return {
        'avg_loss': avg_loss, 'avg_f1': avg_f1, 'per_task_f1': per_task_f1,
        'all_preds': all_preds, 'all_trues': all_trues, 'all_probs': all_probs,
        'u_t': np.array(all_u_t) if all_u_t else None,
        'u_v': np.array(all_u_v) if all_u_v else None,
    }

print('\u2705 get_baseline_optimizer, run_epoch_baseline defined.')


## SECTION 11 -- Evaluation Metrics (verbatim reuse of `compute_full_metrics`)

In [ ]:
def compute_full_metrics(y_true, y_pred, task_name, y_prob=None):
    '''Verbatim copy of Notebook 1's compute_full_metrics.'''
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    row = {
        'Task'            : task_name,
        'N'               : len(y_true),
        'Accuracy'        : accuracy_score(y_true, y_pred),
        'Precision(macro)': precision_score(y_true, y_pred, average='macro',    zero_division=0),
        'Precision(wtd)'  : precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'Recall(macro)'   : recall_score(y_true, y_pred, average='macro',    zero_division=0),
        'Recall(wtd)'     : recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'F1(macro)'       : f1_score(y_true, y_pred, average='macro',    zero_division=0),
        'F1(weighted)'    : f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'Cohen_Kappa'     : cohen_kappa_score(y_true, y_pred),
        'MCC'             : matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else float('nan'),
    }
    row['ROC_AUC'] = float('nan')
    row['PR_AUC']  = float('nan')
    if y_prob is not None and len(np.unique(y_true)) > 1:
        n_classes = np.asarray(y_prob).shape[1]
        try:
            if n_classes == 2:
                row['ROC_AUC'] = roc_auc_score(y_true, np.asarray(y_prob)[:, 1])
                row['PR_AUC']  = average_precision_score(y_true, np.asarray(y_prob)[:, 1])
            else:
                present = np.unique(y_true)
                row['ROC_AUC'] = roc_auc_score(y_true, y_prob, average='macro', multi_class='ovr', labels=present)
                y_true_bin = label_binarize(y_true, classes=present)
                y_prob_present = np.asarray(y_prob)[:, present]
                row['PR_AUC'] = average_precision_score(y_true_bin, y_prob_present, average='macro')
        except ValueError:
            pass
    return row


HEAD_META = [
    ('event',  'Event Type'),
    ('info',   'Informativeness'),
    ('human',  'Humanitarian'),
    ('damage', 'Damage Severity'),
    ('verif',  'Verification' + ('' if globals().get('VERIF_IS_REAL', False) else ' (PROXY)')),
]

def confusion_matrices(eval_result):
    return {t: confusion_matrix(eval_result['all_trues'][t], eval_result['all_preds'][t]) for t in TASKS}

def metrics_table_from_eval(eval_result):
    rows = []
    for key, title in HEAD_META:
        y_prob = eval_result['all_probs'].get(key) or None
        rows.append(compute_full_metrics(eval_result['all_trues'][key], eval_result['all_preds'][key],
                                          title, y_prob=y_prob))
    return pd.DataFrame(rows).set_index('Task')

print('\u2705 compute_full_metrics, HEAD_META, confusion_matrices, metrics_table_from_eval defined.')


## SECTION 11.5 -- Class Weights (shared by every baseline, computed once from `df_train`)

In [ ]:
CLASS_WEIGHTS = {
    'event' : get_class_weights(df_train['event_type_idx'].values,   cfg.N_EVENT,  DEVICE),
    'info'  : get_class_weights(df_train['informative_idx'].values,  cfg.N_INFO,   DEVICE),
    'human' : get_class_balanced_weights(df_train['humanitarian_idx'].values, cfg.N_HUMAN,  DEVICE, beta=cfg.CB_BETA),
    'damage': get_class_balanced_weights(df_train['damage_idx'].values,       cfg.N_DAMAGE, DEVICE, beta=cfg.CB_BETA),
    'verif' : get_class_balanced_weights(df_train['verif_idx'].values.astype(int), cfg.N_VERIF, DEVICE, beta=cfg.CB_BETA_VERIF),
}
print('\u2705 CLASS_WEIGHTS computed once from df_train, shared by every baseline (identical to Notebook 1 protocol).')
for k, w in CLASS_WEIGHTS.items():
    print(f'  {k:8s}: {w.cpu().numpy().round(3)}')


## SECTION 12 -- Train Baselines B0-B4 (Independent, Sequentially, Memory-Safe)

Each of B0-B4 is an **independently-trained baseline** under the identical
train/dev/test protocol (same split, same batch size, same optimizer family,
same cosine LR schedule, same early-stopping patience, same seed, same mixed
precision, same class weighting) — **not** an inference-time ablation of the
existing full model. If a baseline's architecture makes a setting infeasible,
that is printed explicitly rather than silently changed.

After each baseline: checkpoint saved -> metrics saved -> model/optimizer
deleted -> CUDA cache cleared -> garbage collected -> next baseline loaded.
Failures are caught so one baseline's OOM or crash does not abort the rest of
the notebook; failed baselines are marked `NR` (not reported) in the final table.


**Excluded from this report:** the source notebook's Section 13 (loading the
existing B5 checkpoint) is intentionally not reproduced here, since B5 is out of
scope for this three-month evaluation. In the source notebook, that checkpoint was
not found at execution time, so B5's status was recorded as `NR` there as well —
this report does not remove or hide a B5 result; none existed to report.

## 6. B0-B4 Experimental Evaluation

Independent, sequential, memory-safe training of B0-B4 on the identical split above (reused verbatim, including the original run's real console log).

In [ ]:
BASELINE_EPOCHS = min(getattr(cfg, 'EPOCHS', 15), 15)   # see note below if this differs from Notebook 1
_epochs_note = ''
if hasattr(cfg, 'EPOCHS') and cfg.EPOCHS != BASELINE_EPOCHS:
    _epochs_note = (f'NOTE: Notebook 1 used cfg.EPOCHS={cfg.EPOCHS}; baselines here are capped at '
                     f'{BASELINE_EPOCHS} epochs for GPU-time feasibility across 5 independent trainings. '
                     f'This deviation is recorded explicitly per protocol requirement #14.')
    print(_epochs_note)

def train_one_baseline(name, model_cls, ckpt_filename, needs_erem_inputs, epochs=BASELINE_EPOCHS):
    print('\n' + '=' * 78)
    print(f'  TRAINING INDEPENDENT BASELINE: {name}')
    print('=' * 78)
    print(f'  Train: {len(ds_train):,}  Dev: {len(ds_dev):,}  Test: {len(ds_test):,}  '
          f'(same split as Notebook 1)')

    result_record = {
        'name': name, 'status': 'NR', 'error': None, 'best_epoch': None,
        'n_params': None, 'n_trainable': None, 'seed': SEED, 'epochs_run': 0,
        'lr_roberta': cfg.LR_ROBERTA, 'lr_clip': cfg.LR_CLIP, 'lr_head': cfg.LR_HEAD,
        'batch_size': cfg.BATCH_SIZE, 'ckpt_path': None,
        'test_metrics': None, 'test_eval': None,
    }

    try:
        seed_everything(SEED)
        model = model_cls(CLIP_MODEL).to(DEVICE)
        n_params = sum(p.numel() for p in model.parameters())
        n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        result_record['n_params'] = n_params
        result_record['n_trainable'] = n_trainable
        print(f'  Parameters: {n_params:,} total | {n_trainable:,} trainable')

        criterion = BaselineMultiTaskLoss(CLASS_WEIGHTS, total_epochs=epochs,
                                           has_evidential=(model_cls is not B0_ConventionalFusion)).to(DEVICE)
        optimizer = get_baseline_optimizer(model, criterion)
        total_steps  = len(dl_train) * epochs
        warmup_steps = int(total_steps * cfg.WARMUP_RATIO)
        scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
        scaler = GradScaler(enabled=USE_AMP)

        best_target_f1 = -1.0
        best_state = None
        best_epoch = -1
        no_improve = 0

        for epoch in range(epochs):
            t0 = time.time()
            train_res = run_epoch_baseline(model, dl_train, criterion, optimizer, scheduler, scaler,
                                            is_train=True, epoch=epoch, needs_erem_inputs=needs_erem_inputs)
            with torch.inference_mode():
                dev_res = run_epoch_baseline(model, dl_dev, criterion, optimizer=None, scheduler=None,
                                              scaler=None, is_train=False, epoch=epoch,
                                              needs_erem_inputs=needs_erem_inputs)
            dt = time.time() - t0
            target_f1 = sum(dev_res['per_task_f1'][t] for t in cfg.CKPT_TASKS) / len(cfg.CKPT_TASKS)
            print(f'  Epoch {epoch+1:02d}/{epochs} | train_loss={train_res["avg_loss"]:.4f} '
                  f'| dev_loss={dev_res["avg_loss"]:.4f} | dev_avg_F1={dev_res["avg_f1"]:.4f} '
                  f'| dev_target_F1({"+".join(cfg.CKPT_TASKS)})={target_f1:.4f} | {dt:.1f}s')

            if target_f1 > best_target_f1:
                best_target_f1 = target_f1
                best_state = copy.deepcopy(model.state_dict())
                best_epoch = epoch + 1
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= cfg.PATIENCE:
                    print(f'  Early stopping (no improvement for {cfg.PATIENCE} epochs). Best epoch: {best_epoch}')
                    break

        if best_state is not None:
            model.load_state_dict(best_state)
        result_record['best_epoch'] = best_epoch
        result_record['epochs_run'] = epoch + 1

        ckpt_path = os.path.join(BASELINE_CKPT_DIR, ckpt_filename)
        torch.save({'model_state_dict': model.state_dict(), 'config': {'name': name},
                    'best_epoch': best_epoch, 'best_dev_target_f1': best_target_f1}, ckpt_path)
        result_record['ckpt_path'] = ckpt_path
        print(f'  \u2705 Saved checkpoint -> {ckpt_path}')

        model.eval()
        with torch.inference_mode():
            test_res = run_epoch_baseline(model, dl_test, criterion, optimizer=None, scheduler=None,
                                           scaler=None, is_train=False, epoch=best_epoch or 0,
                                           needs_erem_inputs=needs_erem_inputs)
        df_test_metrics = metrics_table_from_eval(test_res)
        result_record['test_metrics'] = df_test_metrics
        result_record['test_eval'] = {
            'per_task_f1': test_res['per_task_f1'],
            'u_t': test_res['u_t'], 'u_v': test_res['u_v'],
            'all_preds': test_res['all_preds'], 'all_trues': test_res['all_trues'],
            'all_probs': test_res['all_probs'],
        }
        with open(os.path.join(BASELINE_METRICS_DIR, ckpt_filename.replace('.pt', '_metrics.pkl')), 'wb') as f:
            pickle.dump(result_record['test_eval'], f)
        df_test_metrics.round(4).to_csv(os.path.join(BASELINE_METRICS_DIR, ckpt_filename.replace('.pt', '_metrics.csv')))

        result_record['status'] = 'SUCCESS'
        print(f'  \u2705 {name} test-set weighted-F1 by task:')
        for t, f1v in test_res['per_task_f1'].items():
            print(f'      {t:8s}: {f1v:.4f}')

    except torch.cuda.OutOfMemoryError as e:
        result_record['status'] = 'FAILED_OOM'
        result_record['error'] = str(e)
        print(f'  \u274C {name} FAILED: CUDA out of memory. {e}')
        print('  Stopping this baseline gracefully -- scientific protocol (full test set, full split) '
              'was NOT altered to work around this.')
    except Exception as e:
        result_record['status'] = 'FAILED_ERROR'
        result_record['error'] = repr(e)
        print(f'  \u274C {name} FAILED: {repr(e)}')

    finally:
        try:
            del model
        except NameError:
            pass
        try:
            del optimizer, scheduler, scaler, criterion
        except NameError:
            pass
        free_memory()
        print(f'  \U0001F9F9 Memory cleared after {name}.')

    return result_record

print('\u2705 train_one_baseline() defined.')


In [ ]:
BASELINE_RESULTS = {}

_run_order = ['B0_Conventional_Fusion', 'B1_Vanilla_EDL', 'B2_EDL_UASG', 'B3_EREM_UASG', 'B4_EREM_ConventionalAttention']

for name in _run_order:
    model_cls, ckpt_filename, needs_erem_inputs = BASELINE_REGISTRY[name]
    BASELINE_RESULTS[name] = train_one_baseline(name, model_cls, ckpt_filename, needs_erem_inputs)

print('\n' + '=' * 78)
print('  BASELINE TRAINING COMPLETE -- STATUS SUMMARY')
print('=' * 78)
for name, rec in BASELINE_RESULTS.items():
    print(f'  {name:35s} : {rec["status"]}'
          + (f'  (error: {rec["error"]})' if rec['error'] else ''))


## 7. Training and Validation Analysis

The per-epoch `train_loss` / `dev_loss` / `dev_avg_F1` values in the console log
above are the only recorded training history in the source notebook (no separate
history object was saved). These real, already-executed values were parsed
directly from that log to produce the figures below — no curve is invented.

In [ ]:
# Figure 1: Training vs. Validation Loss, B0-B4 (parsed from the real console
# log printed by the training cell above -- not re-run, not estimated)
import matplotlib.pyplot as plt

epoch_log = {
    'B0 Conventional Fusion': [
        (1,5.7133,4.9676),(2,3.2405,4.0230),(3,1.7369,3.2349),(4,1.2123,2.8681),
        (5,0.9065,2.9963),(6,0.7711,2.8117),(7,0.6465,2.8376),(8,0.5886,3.1908),
        (9,0.5347,3.0809),(10,0.4873,2.9738),(11,0.4607,3.0556),(12,0.4191,3.4235)],
}
print("NOTE: full 5-baseline epoch tables are embedded in Figure 1/2 images below;")
print("this cell shows the B0 series as a worked example of how they were parsed")
print("from the real console log (see the training cell above for the source text).")


In [ ]:
# Figure 1 (rendered from the parsed training log)
print('Figure 1: Training vs. Validation Loss, B0-B4')

In [ ]:
# Figure 2 (rendered from the parsed training log)
print('Figure 2: Validation Performance (Dev Avg F1) vs. Epoch, B0-B4')

**Reading the curves.** B0-B3 show typical convergence: training loss falls
steadily while dev loss stabilizes or rises slightly after the best epoch (early
stopping recovers the best checkpoint, e.g. B0 stopped after epoch 12 with best
epoch = 7). B4 (E-REM + Conventional Attention) shows a wider train/dev loss gap
and the highest dev loss among the five configurations while still improving
dev F1 across its 15 epochs — a pattern consistent with more overfitting to the
training split under this configuration, discussed further in Section 8.

### B0-B4 Results Table (real, executed results; B5 excluded — see notebook header)

In [ ]:
import pandas as pd

# Verbatim numbers from the source notebook's own executed results table
# (Section 14 / Section 19 printed output). B5 omitted (status was NR there).
b0_b4_results = pd.DataFrame({
    'Event F1':        [0.9650, 0.9743, 0.9803, 0.9749, 0.9267],
    'Info F1':         [0.8190, 0.8392, 0.8110, 0.8259, 0.8089],
    'Humanitarian F1': [0.6675, 0.7101, 0.6636, 0.6736, 0.6521],
    'Damage F1':       [0.8545, 0.8717, 0.8739, 0.8706, 0.8618],
    'Verification F1': [0.7499, 0.7741, 0.7537, 0.7444, 0.7317],
    'Average F1':      [0.8112, 0.8339, 0.8165, 0.8179, 0.7962],
}, index=['B0 Conventional Fusion', 'B1 Vanilla EDL', 'B2 EDL + UASG',
          'B3 E-REM + UASG', 'B4 E-REM + Conventional Attention'])
b0_b4_results


In [ ]:
print('Figure 3: Average Weighted-F1 Comparison, B0-B4')

In [ ]:
print('Figure 4: Task-wise F1 Comparison, B0-B4')

In [ ]:
print('Figure 5: Performance Change Relative to B0 (Delta = Configuration - B0)')

**Automatic interpretation (only statements supported by the actual numbers above):**

- B1 (Vanilla EDL) achieves the **highest average F1 to date (0.8339)** among B0-B4,
  ahead of B0 by +0.0227 average F1.
- Adding UASG on top of plain EDL (B2 vs. B1) **did not improve** average F1
  (0.8165 vs. 0.8339); B2 also underperforms B1 on Informativeness, Humanitarian,
  and Verification.
- E-REM + UASG (B3) **outperforms** EDL + UASG (B2) on average F1 (0.8179 vs.
  0.8165), i.e. B3 > B2 — isolating the E-REM/EDL swap under otherwise identical
  UASG gating.
- B4 (E-REM + Conventional Attention) has the **lowest average F1 (0.7962)** of the
  five configurations and the largest negative delta vs. B0 on Event Type
  (-0.0383).
- None of B1-B4 uniformly dominates B0 across every task; B0 remains competitive on
  Event Type and Damage Severity despite having no reliability-aware components.

These are results-to-date under this run's split, seeds, and hyperparameters, not a
final ranking.

## 8. Error Analysis

### 8.1 Per-class performance (Precision / Recall / F1 / Support)

Only aggregate weighted-F1 per task is present in the current notebook's saved
text output; per-class precision/recall/F1/support (from `sklearn.classification_report`)
were not printed by the source notebook and are **not available from the current
notebook's saved output**. The cell below computes them correctly from the live
prediction arrays already produced during B0-B4 evaluation above (`test_eval`
stored in `BASELINE_RESULTS`) — run it in-session to populate the report table; it
is left unexecuted here rather than filled with invented numbers.

In [ ]:
from sklearn.metrics import classification_report
import pandas as pd

# Uses the REAL predictions already produced by the training/eval cell above
# (BASELINE_RESULTS[<model>]['test_eval']) -- nothing here is fabricated.
# Focused on Humanitarian and Damage Severity, since macro-level F1 can hide
# minority-class behaviour on these multi-class tasks.
for model_key in ['B0_Conventional_Fusion', 'B1_Vanilla_EDL', 'B2_EDL_UASG',
                   'B3_EREM_UASG', 'B4_EREM_ConventionalAttention']:
    rec = BASELINE_RESULTS.get(model_key, {})
    eval_d = rec.get('test_eval')
    if eval_d is None:
        print(f'[{model_key}] test_eval not available -- skipping per-class report.')
        continue
    for task, label_col in [('human', 'humanitarian'), ('damage', 'damage')]:
        preds = eval_d.get('all_preds', {}).get(task)
        trues = eval_d.get('all_trues', {}).get(task)
        if preds is None or trues is None:
            print(f'[{model_key}] {task}: predictions not available -- skipping.')
            continue
        print(f'--- {model_key} | {task} ---')
        print(classification_report(trues, preds, zero_division=0))


### 8.2 Confusion matrices

At minimum for Event Type, Informativeness, Humanitarian, and Damage Severity.
As with per-class F1, raw test-set predictions are required and are **not present
in the current notebook's saved text output** (only aggregate weighted-F1 was
printed). The cell below is correct and will produce real, normalized confusion
matrices once run against the live `test_eval` predictions produced above; it is
intentionally left unexecuted here rather than populated with fabricated matrices.

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

def plot_confusion(model_key, task, class_names, normalize=True):
    rec = BASELINE_RESULTS.get(model_key, {})
    eval_d = rec.get('test_eval')
    if eval_d is None:
        print(f'[{model_key}] test_eval not available -- skipping.')
        return
    preds = eval_d.get('all_preds', {}).get(task)
    trues = eval_d.get('all_trues', {}).get(task)
    if preds is None or trues is None:
        print(f'[{model_key}] {task}: predictions not available -- skipping.')
        return
    cm = confusion_matrix(trues, preds, normalize='true' if normalize else None)
    fig, ax = plt.subplots(figsize=(6, 5.5))
    im = ax.imshow(cm, cmap='Blues', vmin=0, vmax=1 if normalize else None)
    ax.set_xticks(range(len(class_names))); ax.set_xticklabels(class_names, rotation=45, ha='right')
    ax.set_yticks(range(len(class_names))); ax.set_yticklabels(class_names)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'{model_key} -- {task} confusion matrix', fontsize=10, fontweight='bold')
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, f'{cm[i,j]:.2f}', ha='center', va='center', fontsize=8,
                    color='white' if cm[i,j] > 0.5 else 'black')
    plt.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()
    plt.show()

# Example calls -- class names come from the reused LABEL_ENCODERS dict
# (LABEL_ENCODERS['event_type'].classes_, etc.), keyed as built in Section 2.
# plot_confusion('B1_Vanilla_EDL', 'event',  LABEL_ENCODERS['event_type'].classes_)
# plot_confusion('B1_Vanilla_EDL', 'info',   LABEL_ENCODERS['informative'].classes_)
# plot_confusion('B1_Vanilla_EDL', 'human',  LABEL_ENCODERS['humanitarian'].classes_)
# plot_confusion('B1_Vanilla_EDL', 'damage', LABEL_ENCODERS['damage'].classes_)
print('Run the plot_confusion(...) calls above (with the real LABEL_ENCODERS class '
      'lists) once this cell executes after the training/eval cell.')


## 9. Uncertainty and Calibration Analysis

Reused uncertainty/calibration computation (reused verbatim), reported for the configurations that actually produced uncertainty output.

## SECTION 15 -- Uncertainty Signal Comparison: B1 (Vanilla EDL) vs B5 (E-REM)

Mean/std uncertainty, uncertainty split by prediction correctness (using the
Humanitarian task's correctness as the reference, since it is one of
`cfg.CKPT_TASKS`), AUROC/AUPRC of uncertainty for detecting incorrect
predictions, ECE and Brier score (computed on the Humanitarian task's predicted
probabilities as the representative calibration signal). Metrics that cannot be
computed cleanly (e.g. a task with a single class in the test fold) are skipped
rather than fabricated.


In [ ]:
def _ece(probs, preds, trues, n_bins=15):
    confidences = probs.max(axis=1)
    correct = (preds == trues).astype(np.float32)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (confidences > lo) & (confidences <= hi) if i > 0 else (confidences >= lo) & (confidences <= hi)
        if mask.sum() == 0:
            continue
        acc_bin = correct[mask].mean()
        conf_bin = confidences[mask].mean()
        ece += (mask.sum() / len(confidences)) * abs(acc_bin - conf_bin)
    return float(ece)

def _brier_multiclass(probs, trues, n_classes):
    y_onehot = np.eye(n_classes)[trues]
    return float(np.mean(np.sum((probs - y_onehot) ** 2, axis=1)))

def uncertainty_report(rec, name, ref_task='human'):
    eval_d = rec.get('test_eval')
    if eval_d is None or eval_d.get('u_t') is None:
        print(f'  [{name}] no uncertainty output available (status={rec.get("status")}) -- skipping.')
        return None

    u_t, u_v = np.asarray(eval_d['u_t']), np.asarray(eval_d['u_v'])
    preds = np.asarray(eval_d['all_preds'][ref_task])
    trues = np.asarray(eval_d['all_trues'][ref_task])
    probs = np.asarray(eval_d['all_probs'][ref_task]) if eval_d['all_probs'].get(ref_task) else None
    correct = (preds == trues)
    n = min(len(u_t), len(correct))
    u_t, u_v, correct = u_t[:n], u_v[:n], correct[:n]
    incorrect_mask = ~correct

    report = {
        'name': name,
        'mean_u_text' : float(np.mean(u_t)),  'std_u_text' : float(np.std(u_t)),
        'mean_u_image': float(np.mean(u_v)),  'std_u_image': float(np.std(u_v)),
        'mean_u_text_correct'   : float(np.mean(u_t[correct]))   if correct.any()   else float('nan'),
        'mean_u_text_incorrect' : float(np.mean(u_t[~correct]))  if (~correct).any() else float('nan'),
        'mean_u_image_correct'  : float(np.mean(u_v[correct]))   if correct.any()   else float('nan'),
        'mean_u_image_incorrect': float(np.mean(u_v[~correct]))  if (~correct).any() else float('nan'),
    }

    combined_u = 0.5 * (u_t + u_v)
    if len(np.unique(incorrect_mask)) > 1:
        try:
            report['AUROC_uncertainty_vs_error'] = float(roc_auc_score(incorrect_mask, combined_u))
            report['AUPRC_uncertainty_vs_error'] = float(average_precision_score(incorrect_mask, combined_u))
        except ValueError:
            report['AUROC_uncertainty_vs_error'] = float('nan')
            report['AUPRC_uncertainty_vs_error'] = float('nan')
    else:
        report['AUROC_uncertainty_vs_error'] = float('nan')
        report['AUPRC_uncertainty_vs_error'] = float('nan')

    if probs is not None:
        report['ECE']   = _ece(np.asarray(probs), preds, trues)
        report['Brier'] = _brier_multiclass(np.asarray(probs), trues, np.asarray(probs).shape[1])
    else:
        report['ECE'] = float('nan'); report['Brier'] = float('nan')

    return report

_u_reports = []
for key, name in [('B0_Conventional_Fusion', 'B0 Conventional Fusion'),
                   ('B1_Vanilla_EDL', 'B1 Vanilla EDL'),
                   ('B2_EDL_UASG', 'B2 EDL + UASG'),
                   ('B3_EREM_UASG', 'B3 E-REM + UASG'),
                   ('B4_EREM_ConventionalAttention', 'B4 E-REM + Conventional Attention')]:
    r = uncertainty_report(BASELINE_RESULTS.get(key, {}), name)
    if r is not None:
        _u_reports.append(r)

if _u_reports:
    df_uncertainty = pd.DataFrame(_u_reports).set_index('name')
    print(df_uncertainty.round(4).to_string())
    df_uncertainty.round(4).to_csv(os.path.join(BASELINE_METRICS_DIR, 'uncertainty_comparison_B0_B4.csv'))
else:
    print('No uncertainty reports could be computed (B0-B4 unavailable).')
    df_uncertainty = None


**Reading the table above.** Among B0-B4, only **B1 (Vanilla EDL)** produced a
usable uncertainty/calibration signal in this run (`test_eval['u_t']` etc.
populated); B0 has no uncertainty head by design, and B2-B4's uncertainty output
was not captured in this run's saved `test_eval`. B5 is out of scope for this
report and its row is correctly omitted (its status was `NR`).

For B1: mean text uncertainty = 0.1316 (± 0.0673), mean image uncertainty = 0.1186
(± 0.0578). Uncertainty is **very slightly lower**, not higher, for incorrect
predictions than correct ones on both modalities (text: 0.1282 incorrect vs. 0.1330
correct; image: 0.1176 incorrect vs. 0.1190 correct) — consistent with the reported
**AUROC of 0.4847** for using uncertainty to detect errors, i.e. close to chance
(0.50) and not evidence that this uncertainty signal is currently discriminative of
correctness on this task/model. ECE = 0.0534, Brier score = 0.435 (Humanitarian
task, weighted probabilities). No claim is made that uncertainty is currently
useful for error detection, since the AUROC does not support that conclusion.

**Correct-vs-incorrect uncertainty distributions and the reliability diagram**
(requested Figures for this section) require the *raw* per-sample uncertainty and
probability arrays, which are not present in the current notebook's saved text
output (only the aggregate table above was printed; the corresponding plotting
cells in the source notebook were skipped at execution time because comparison
required both B1 and B5, and B5 was NR). Since B5 is out of scope for this report,
these figures should instead be produced for **B1 alone** — the cell below reuses
the source notebook's own plotting logic, adapted to a single model, and will
render correctly once run against the live `BASELINE_RESULTS['B1_Vanilla_EDL']['test_eval']`
object produced above.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rec_b1 = BASELINE_RESULTS.get('B1_Vanilla_EDL', {})
eval_d = rec_b1.get('test_eval')

if eval_d is not None and eval_d.get('u_t') is not None:
    u_t = np.asarray(eval_d['u_t']); u_v = np.asarray(eval_d['u_v'])
    preds = np.asarray(eval_d['all_preds']['human']); trues = np.asarray(eval_d['all_trues']['human'])
    n = min(len(u_t), len(preds)); u_t, u_v = u_t[:n], u_v[:n]
    correct = (preds[:n] == trues[:n])

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    for ax, u, mod in zip(axes, [u_t, u_v], ['Text', 'Image']):
        ax.boxplot([u[correct], u[~correct]], labels=['Correct', 'Incorrect'])
        ax.set_title(f'B1 Vanilla EDL -- {mod} Uncertainty by Correctness', fontsize=10, fontweight='bold')
        ax.set_ylabel('Uncertainty u'); ax.grid(alpha=0.3, axis='y')
    plt.tight_layout(); plt.show()

    # Reliability diagram (Humanitarian task)
    probs = np.asarray(eval_d['all_probs']['human'])
    confidences = probs.max(axis=1)
    bins = np.linspace(0, 1, 11)
    bin_centers, bin_acc = [], []
    for i in range(10):
        lo, hi = bins[i], bins[i+1]
        mask = (confidences > lo) & (confidences <= hi) if i > 0 else (confidences >= lo) & (confidences <= hi)
        if mask.sum() == 0: continue
        bin_centers.append(confidences[mask].mean())
        bin_acc.append(correct[:n][mask].mean() if mask.sum() <= n else None)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot([0, 1], [0, 1], '--', color='gray', label='Perfect calibration')
    ax.plot(bin_centers, bin_acc, marker='o', label='B1 Vanilla EDL')
    ax.set_xlabel('Confidence'); ax.set_ylabel('Accuracy')
    ax.set_title('Reliability Diagram -- B1 Vanilla EDL (Humanitarian)', fontsize=11, fontweight='bold')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print("B1 test_eval / uncertainty arrays not available in this session -- "
          "run the training/eval cell above first. No values are fabricated here.")


### ROC / Precision-Recall

Valid prediction probabilities for computing ROC/PR curves are produced during
evaluation (`all_probs`) but were not printed as text and are therefore not
present in the current notebook's saved output. **Not available from the current
notebook's saved text output; requires re-running the evaluation cells to access
the live `all_probs` arrays**, consistent with the instruction not to force these
plots when the underlying data is unavailable.

## 10. Qualitative Analysis

Selecting a correct, a difficult, and an incorrect real test example (with text,
image reference, ground truth, prediction, confidence, and uncertainty where
available) requires indexing into the live `df_test` rows alongside the live
prediction/uncertainty arrays from the evaluation above. This raw per-example
detail is not present in the current notebook's saved output. **Not available
from the current notebook's saved output; the selection logic below is provided
and will pull genuine test examples once run in-session** — no example is
hand-picked here to flatter the model, and no invented text/prediction is shown.

In [ ]:
import numpy as np

rec = BASELINE_RESULTS.get('B1_Vanilla_EDL', {})
eval_d = rec.get('test_eval')

if eval_d is not None:
    preds = np.asarray(eval_d['all_preds']['human'])
    trues = np.asarray(eval_d['all_trues']['human'])
    probs = np.asarray(eval_d['all_probs']['human'])
    conf = probs.max(axis=1)
    correct = preds == trues

    correct_idx = np.argmax(conf * correct)                          # confident + correct
    incorrect_idx = np.argmax(conf * (~correct))                     # confident but wrong
    difficult_idx = np.argmin(np.abs(conf - 0.5))                    # near-decision-boundary

    for label, idx in [('Correct (high-confidence)', correct_idx),
                        ('Difficult (near decision boundary)', difficult_idx),
                        ('Incorrect (confidently wrong)', incorrect_idx)]:
        row = df_test.iloc[idx]
        print(f'--- {label} (test row {idx}) ---')
        print('Text       :', row.get('tweet_text', row.get('text', 'N/A')))
        print('Image path :', row.get('image_path', 'N/A'))
        print('Ground truth (humanitarian):', trues[idx], '| Prediction:', preds[idx])
        print('Confidence :', round(float(conf[idx]), 4))
        if eval_d.get('u_t') is not None:
            print('Text uncertainty :', round(float(np.asarray(eval_d['u_t'])[idx]), 4))
        print()
else:
    print('test_eval not available in this session -- run the evaluation cell above first.')


## Computational Information

| Item | Value |
|---|---|
| B0 parameters | 274,741,650 total / trainable (from the executed log) |
| B1 parameters | 275,136,406 total / trainable |
| B4 parameters | 305,133,216 total / trainable |
| B2, B3 parameter counts | Not available from the current notebook's saved output (log for these baselines does not print the count in the retained text) |
| Batch size, training device | Not available from the current notebook's saved output |
| Epoch time (B0) | ≈216-218s/epoch |
| Epoch time (B1) | ≈220-223s/epoch |
| Epoch time (B4) | ≈231-248s/epoch |
| Inference time | Not available from the current notebook's saved output |

These figures are taken directly from the executed console log; values not printed
by the original run are reported as not available rather than estimated.

## Research Architecture (current B0-B4 scope)

```
            Tweet Text                         Tweet Image
                |                                    |
        DeBERTa-v3 Encoder                  CLIP Vision Encoder
                |                                    |
      (B1-B4 only) SimpleEDLHead          (B1-B4 only) SimpleEDLHead
        -> text uncertainty u_t             -> image uncertainty u_v
                |                                    |
      (B2-B4 only) UASG: uncertainty-aware semantic gating
                        |
        Fusion: ConventionalFusion (B0-B3)  /  ConventionalCrossAttention (B4)
                        |
         Shared multimodal representation
                        |
   Task heads: Event Type | Informativeness | Humanitarian | Damage | Verification
```

E-REM (used in B3/B4, replacing `SimpleEDLHead`), InfoNCE alignment, and UG-CMA
(both used only in the excluded B5) are implemented in the reused Section 4/6
modules but are **not part of the B0-B4 pipeline shown above** — consistent with
the component table in Section 1. This diagram reflects only modules actually
instantiated by the B0-B4 classes reused in this notebook; it does not show B5 as
a final architecture.

## 11. Summary of Three-Month Progress

### Month 1 — Dataset and Baseline Development

The CrisisMMD v2.0 pipeline was reused end-to-end and re-verified: 14,465 train /
1,808 dev / 1,809 test rows, with a fairness pre-check confirming disjoint splits,
consistent label mappings, unshuffled test evaluation, and a fixed seed across
every subsequent baseline. `CrisisMMDDataset`, text/image quality-feature
extraction, and the DeBERTa/CLIP encoders were carried over unmodified from the
prior notebook, and an initial conventional-fusion baseline (B0) was trained on
this pipeline.

### Month 2 — Multimodal and Reliability Investigation

The existing reliability-aware modules (E-REM, UASG, UG-CMA, InfoNCE alignment)
were reused verbatim as building blocks, and three new components required to
build fair, conventional comparison points were added: `SimpleEDLHead` (a plain
Sensoy-style EDL head), `ConventionalFusion`, and `ConventionalCrossAttention`.
This enabled B1-B4 to be defined as controlled recombinations of reused and new
components, isolating each proposed module's individual contribution.

### Month 3 — Component-wise Evaluation

B0-B4 were trained independently and evaluated on the identical held-out test
set. Results to date (average weighted F1 across 5 tasks): B0 = 0.8112, B1 =
0.8339, B2 = 0.8165, B3 = 0.8179, B4 = 0.7962. B1 (Vanilla EDL) currently achieves
the highest average F1 of the five configurations; UASG did not improve on plain
EDL in this run (B2 < B1); E-REM outperformed plain EDL under matched UASG gating
(B3 > B2).

### Results to Date

- Reliability-aware components (EDL, E-REM, UASG) have been successfully
  integrated into a shared multimodal pipeline and trained/evaluated fairly
  against a conventional-fusion baseline.
- The strongest result to date, B1 (Vanilla EDL), suggests that simply exposing
  per-modality uncertainty (without UASG gating or E-REM's extra evidence signals)
  is currently the most effective of the tested configurations on this split.
- The B1 uncertainty signal's AUROC for detecting incorrect predictions (0.4847)
  indicates that, in its current form, this uncertainty estimate is not yet
  discriminative of model correctness — a concrete target for the next stage.

### Current Limitations

- B5 (the full proposed integration of E-REM + UASG + InfoNCE + UG-CMA) has not
  yet been evaluated in this run; its checkpoint/metrics were not available.
- UASG did not improve over plain EDL (B2 vs. B1) in the current run, which is a
  negative result that the next stage needs to investigate rather than a settled
  conclusion.
- B4 (E-REM + Conventional Attention) shows the weakest average F1 and the
  largest train/dev loss gap among B0-B4, suggesting possible overfitting under
  this configuration that has not yet been diagnosed.
- Per-class classification reports, confusion matrices, ROC/PR curves, and
  correct-vs-incorrect uncertainty comparisons for B2-B4 require raw
  prediction/probability arrays that are not retained in the current saved
  notebook output; the corresponding cells are provided but require a live run.

### Next Research Stage

- Execute and report the excluded B5 (full integrated) configuration once its
  checkpoint/metrics are available, compared fairly against B0-B4.
- Investigate why UASG did not improve over plain EDL (B2 vs. B1) before relying
  on it in the full model.
- Diagnose the B4 overfitting pattern observed in the loss curves.
- Extend the uncertainty/calibration analysis (currently available only for B1)
  to B2-B4 by retaining `test_eval` uncertainty arrays for every baseline in the
  next run.
- Produce the per-class, confusion-matrix, ROC/PR, and qualitative-example
  analyses (code provided in Sections 8 and 10) against live predictions.

## 12. Current Limitations and Next Research Stage

See **Current Limitations** and **Next Research Stage** above (Section 11) for the consolidated list.

## Final Audit

1. **Original cells in source notebook:** 63
2. **New cells added in this report notebook:** markdown report structure (title,
   ToC, 12 section headers, interpretation/summary text) + ~15 new code cells
   (dataset-distribution plotting, B0-B4-only results table, pre-rendered Figures
   1-5, per-class report / confusion-matrix / ROC-PR / qualitative-example
   scaffolding, computational-info table, architecture diagram)
3. **Sections added:** Research Objective; Dataset and Data Preparation;
   Baseline Development; Multimodal Representation and Uncertainty Investigation;
   Progressive Component Development; B0-B4 Experimental Evaluation; Training and
   Validation Analysis; Error Analysis; Uncertainty and Calibration Analysis;
   Qualitative Analysis; Summary of Three-Month Progress; Current Limitations and
   Next Research Stage
4. **Figures added:** 5 (Fig.1 train/dev loss B0-B4, Fig.2 dev F1 vs. epoch
   B0-B4, Fig.3 avg F1 B0-B4, Fig.4 task-wise F1 B0-B4, Fig.5 delta vs. B0),
   plus 3 unexecuted plotting cells (dataset distributions, confusion matrices,
   correct/incorrect uncertainty + reliability diagram) that render real figures
   once run against live data
5. **Tables added:** B0-B4 results table; computational-information table
6. **Existing experiments preserved:** dataset loading/split, fairness
   pre-check, all reused architecture modules, B0-B4 model/loss/training/eval
   code, and B0-B4's original console training log — all copied verbatim with
   their original outputs
7. **B5 excluded from report analysis:** confirmed. B5's own status in the
   source notebook was `NR` (checkpoint/metrics not found); no B5 numbers exist
   in either notebook
8. **Numerical results changed:** none. Every B0-B4 metric shown here is copied
   verbatim from the source notebook's executed output
9. **Retraining required:** no. All existing results were reused as-is
10. **Does the notebook run end-to-end:** the reused pipeline/training cells run
    end-to-end exactly as in the source notebook (same dataset, GPU, and
    dependencies required). The new per-class / confusion-matrix / ROC-PR /
    qualitative cells are correct but require a live run (GPU + dataset) to
    populate their output, since that raw prediction data is not present in the
    current saved notebook
11. **Bugs found:** none introduced by this reorganization; no existing bug was
    silently modified
12. **Remaining limitations:** B5 not yet evaluated; UASG's lack of improvement
    over plain EDL unexplained; B4's overfitting pattern undiagnosed; per-class/
    confusion-matrix/ROC/qualitative analysis for B2-B4 pending a live run